> **Note:** originally developed and run in Google Colab. Google-Drive mounting cells were removed, Drive paths were rewritten as relative paths and outputs were stripped (figures of the original run are in `assets/`). The paths/cleanup were **not re-executed** after this change, so expect to adjust paths.

# **Segmentación Semántica**

In [ ]:
# ============================================================
# GlandNet - UNet 3-level + Tversky + Watershed postprocessing
# ============================================================

# Montar Drive

# ---- Instalación de librerías ----
!pip install -q albumentations==1.4.3 scikit-image scipy

# ---- Imports generales ----
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.metrics import hausdorff_distance
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

from tqdm import tqdm

# ============================================================
# 1) Cargar pares imagen-máscara (CSV ya generado)
# ============================================================
base_path = "."
pairs_csv = Path(base_path) / "metadata/train_pairs.csv"

df_pairs = pd.read_csv(pairs_csv)
# Fix: Update paths to use the correct base directory (GlandNet2)
df_pairs["image_path"] = df_pairs["image_path"].str.replace("GlandNet/data", "GlandNet2/data")
df_pairs["mask_path"] = df_pairs["mask_path"].str.replace("GlandNet/data", "GlandNet2/data")
print(df_pairs.head())
print("Total pares:", len(df_pairs))

# ============================================================
# 2) Split train / val
# ============================================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

indices = list(range(len(df_pairs)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx   = indices[split:]

df_train = df_pairs.iloc[train_idx].reset_index(drop=True)
df_val   = df_pairs.iloc[val_idx].reset_index(drop=True)

print("Train:", len(df_train), "Val:", len(df_val))

# ============================================================
# 3) Dataset y transforms
# ============================================================
class GlasSegmentationDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]        # [3,H,W]
            mask  = augmented["mask"]        # [H,W] o [1,H,W]
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)     # [1,H,W]
        else:
            image = image.transpose(2,0,1).astype("float32") / 255.
            image = torch.tensor(image)
            mask  = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask

# ---- Augmentations ----
train_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            p=0.5, border_mode=0
        ),
        # Morfología
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=10, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        # Color / iluminación
        A.HueSaturationValue(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        # Normalización
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

train_ds = GlasSegmentationDataset(df_train, transforms=train_transforms)
val_ds   = GlasSegmentationDataset(df_val,   transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2)

print("len(train_ds), len(val_ds) =", len(train_ds), len(val_ds))

# ============================================================
# 4) Modelo: UNet 3 niveles
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.pool  = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)

        # Decoder
        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        c3 = self.down3(p2)
        p3 = self.pool(c3)

        bn = self.bottleneck(p3)

        u3 = self.up3(bn)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)

        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)

# ============================================================
# 5) Funciones de pérdida
# ============================================================
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2,3))
    den = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
    dice = num / den
    return 1 - dice.mean()

def tversky_loss(logits, targets, alpha=0.7, beta=0.3, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=(2,3))
    fp = (probs * (1 - targets)).sum(dim=(2,3))
    fn = ((1 - probs) * targets).sum(dim=(2,3))
    tversky = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
    return 1 - tversky.mean()

# ============================================================
# 6) Setup entrenamiento
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet().to(device)
bce   = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8
)

patience = 15
epochs_no_improve = 0
best_val_dice = 0.0
num_epochs = 120

models_dir = Path(base_path) / "models"
models_dir.mkdir(exist_ok=True)
best_model_path = models_dir / "best_unet_glas_3lvl_tversky.pth"

train_losses = []
val_dices    = []

# ============================================================
# 7) Loop de entrenamiento
# ============================================================
for epoch in range(1, num_epochs+1):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)

        loss_bce     = bce(logits, masks)
        loss_d       = dice_loss(logits, masks)
        loss_tversky = tversky_loss(logits, masks)

        loss = 0.3 * loss_bce + 0.3 * loss_d + 0.4 * loss_tversky

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)
    train_losses.append(train_loss)

    # --- VALID ---
    model.eval()
    val_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            logits = model(imgs)
            d = 1 - dice_loss(logits, masks).item()
            val_dice += d
            n_batches += 1

    val_dice /= max(n_batches, 1)
    val_dices.append(val_dice)

    scheduler.step(val_dice)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_dice={val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_model_path)
        print(f"  ↳ Nuevo mejor modelo guardado en {best_model_path}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  ↳ Sin mejora en val_dice por {epochs_no_improve} epoch(s)")
        if epochs_no_improve >= patience:
            print(f"🔚 Early stopping activado en epoch {epoch}")
            break

# ============================================================
# 8) Curvas de entrenamiento
# ============================================================
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida de entrenamiento")
plt.grid(True)
plt.legend()

plt.subplot(1,2,2)
plt.plot(val_dices, label="Val Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Dice en validación")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 9) Post-proceso y métricas tipo GlaS
# ============================================================
def clean_mask(pred_bin, min_size=30, opening_radius=0):
    mask = pred_bin.astype(bool)
    mask = remove_small_objects(mask, min_size=min_size)
    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)
    return mask.astype(np.uint8)

def split_glands_watershed(pred_bin, footprint_size=15):
    distance = ndi.distance_transform_edt(pred_bin)
    coords = peak_local_max(
        distance,
        footprint=np.ones((footprint_size, footprint_size)),
        labels=pred_bin
    )
    markers_mask = np.zeros(distance.shape, dtype=bool)
    if coords.size > 0:
        markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    labels_ws = watershed(-distance, markers, mask=pred_bin)
    return labels_ws

def compute_dice(mask1, mask2, eps=1e-6):
    inter = np.logical_and(mask1, mask2).sum()
    union = mask1.sum() + mask2.sum()
    return 2 * inter / (union + eps)

def compute_f1(tp, fp, fn):
    return tp / (tp + 0.5*(fp + fn) + 1e-6)

def object_metrics(pred_mask, gt_mask):
    labeled_pred = label(pred_mask)
    labeled_gt   = label(gt_mask)

    pred_regions = regionprops(labeled_pred)
    gt_regions   = regionprops(labeled_gt)

    num_pred = len(pred_regions)
    num_gt   = len(gt_regions)

    object_dices = []
    hausdorffs   = []

    dice_matrix = np.zeros((num_gt, num_pred))

    for i, gt in enumerate(gt_regions):
        gt_mask_i = (labeled_gt == gt.label)
        for j, pr in enumerate(pred_regions):
            pr_mask_j = (labeled_pred == pr.label)
            d = compute_dice(gt_mask_i, pr_mask_j)
            dice_matrix[i, j] = d

    tp = 0
    fn = 0
    matched_pred = set()

    for i in range(num_gt):
        if num_pred == 0:
            fn += 1
            continue
        j = np.argmax(dice_matrix[i])
        best_dice = dice_matrix[i, j]

        gt_mask_i = (labeled_gt == (i+1))
        pr_mask_j = (labeled_pred == (j+1))

        if best_dice > 0.5:
            tp += 1
            matched_pred.add(j)
            object_dices.append(best_dice)
            hd = hausdorff_distance(gt_mask_i, pr_mask_j)
            hausdorffs.append(hd)
        else:
            fn += 1

    fp = num_pred - len(matched_pred)
    f1 = compute_f1(tp, fp, fn)

    mean_dice = np.mean(object_dices) if len(object_dices) > 0 else 0.0
    mean_hd   = np.mean(hausdorffs)   if len(hausdorffs)   > 0 else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "object_dice_mean": mean_dice,
        "hausdorff_mean":   mean_hd,
        "F1": f1
    }

# ============================================================
# 10) Cargar mejor modelo
# ============================================================
model = UNet().to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("Modelo recargado desde:", best_model_path)

# ============================================================
# 11) Barrido de thresholds en validación
# ============================================================
thresholds = np.linspace(0.3, 0.7, 9)
results_by_thr = []

for thr in thresholds:
    all_results_thr = []

    for imgs, gt_mask in val_loader:
        imgs = imgs.to(device)

        with torch.no_grad():
            pred_logits = model(imgs)
            pred_prob   = torch.sigmoid(pred_logits).cpu().numpy()[0,0]

        pred_bin_raw = (pred_prob > thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        gt_mask_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        res = object_metrics(pred_final, gt_mask_np)
        all_results_thr.append(res)

    mean_dice_thr = np.mean([r["object_dice_mean"] for r in all_results_thr])
    mean_hd_thr   = np.mean([r["hausdorff_mean"]   for r in all_results_thr])
    mean_f1_thr   = np.mean([r["F1"]               for r in all_results_thr])

    results_by_thr.append((thr, mean_dice_thr, mean_hd_thr, mean_f1_thr))
    print(f"thr={thr:.2f} | Dice={mean_dice_thr:.4f} | HD={mean_hd_thr:.2f} | F1={mean_f1_thr:.4f}")

best_thr, best_dice, best_hd, best_f1 = max(results_by_thr, key=lambda x: x[3])
print("\n>>> Mejor threshold según F1:", best_thr)
print(f"    Dice={best_dice:.4f} | HD={best_hd:.2f} | F1={best_f1:.4f}")

# ============================================================
# 12) Métricas globales en VALIDATION
# ============================================================
model.eval()
all_results = []

for imgs, gt_mask in tqdm(val_loader, total=len(val_loader)):
    imgs = imgs.to(device)

    with torch.no_grad():
        pred_logits = model(imgs)
        pred_prob   = torch.sigmoid(pred_logits).cpu().numpy()[0,0]

    pred_bin_raw = (pred_prob > best_thr).astype(np.uint8)
    pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
    labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
    pred_final   = (labels_ws > 0).astype(np.uint8)

    gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

    res = object_metrics(pred_final, gt_np)
    all_results.append(res)

mean_object_dice = np.mean([r["object_dice_mean"] for r in all_results])
mean_hausdorff   = np.mean([r["hausdorff_mean"]   for r in all_results])
mean_f1          = np.mean([r["F1"]               for r in all_results])

print("\n===== MÉTRICAS VALIDACIÓN =====")
print(f"Object Dice promedio: {mean_object_dice:.4f}")
print(f"Hausdorff promedio:   {mean_hausdorff:.2f}")
print(f"F1 object-level:      {mean_f1:.4f}")

# ============================================================
# 13) TEST A/B
# ============================================================
test_images_dir = Path(base_path) / "data/test/images"
test_masks_dir  = Path(base_path) / "data/test/masks"

img_files = sorted([p for p in test_images_dir.iterdir()
                    if p.suffix.lower() == ".bmp"])

pairs_test = []
for img_path in img_files:
    name = img_path.stem              # testA_1, testB_2, ...
    mask_path = test_masks_dir / f"{name}_anno.bmp"
    if mask_path.exists():
        pairs_test.append({
            "image_path": str(img_path),
            "mask_path":  str(mask_path)
        })
    else:
        print("⚠ No se encontró máscara para:", img_path.name)

df_test = pd.DataFrame(pairs_test)
print("Total pares en test:", len(df_test))
df_test.head()

df_test["subset"] = df_test["image_path"].apply(
    lambda x: "A" if "testA" in x else "B"
)

df_test_A = df_test[df_test["subset"]=="A"].reset_index(drop=True)
df_test_B = df_test[df_test["subset"]=="B"].reset_index(drop=True)

class GlasSegTestWithGT(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            image = torch.tensor(image.transpose(2,0,1) / 255.)
            mask  = torch.tensor(mask).unsqueeze(0)

        return image, mask

testA_ds = GlasSegTestWithGT(df_test_A, transforms=val_transforms)
testB_ds = GlasSegTestWithGT(df_test_B, transforms=val_transforms)

testA_loader = DataLoader(testA_ds, batch_size=1, shuffle=False)
testB_loader = DataLoader(testB_ds, batch_size=1, shuffle=False)

def evaluate_GlaS_loader(loader, model, device, best_thr):
    results = []
    model.eval()

    for imgs, gt_mask in tqdm(loader, total=len(loader)):
        imgs = imgs.to(device)
        gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        with torch.no_grad():
            pred = torch.sigmoid(model(imgs)).cpu().numpy()[0,0]

        pred_bin_raw = (pred > best_thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        res = object_metrics(pred_final, gt_np)
        results.append(res)

    mean_dice = np.mean([r["object_dice_mean"] for r in results])
    mean_hd   = np.mean([r["hausdorff_mean"]   for r in results])
    mean_f1   = np.mean([r["F1"]               for r in results])

    return mean_dice, mean_hd, mean_f1

print("\n===== MÉTRICAS TEST A =====")
dice_A, hd_A, f1_A = evaluate_GlaS_loader(testA_loader, model, device, best_thr)
print(f"Object Dice (A): {dice_A:.4f}")
print(f"Hausdorff (A):   {hd_A:.2f}")
print(f"F1 objetual (A): {f1_A:.4f}")

print("\n===== MÉTRICAS TEST B =====")
dice_B, hd_B, f1_B = evaluate_GlaS_loader(testB_loader, model, device, best_thr)
print(f"Object Dice (B): {dice_B:.4f}")
print(f"Hausdorff (B):   {hd_B:.2f}")
print(f"F1 objetual (B): {f1_B:.4f}")

print("\n========== RESUMEN FINAL TEST A/B ==========")
print("Subset | Dice | Hausdorff | F1")
print("------------------------------------------")
print(f"Test A | {dice_A:.4f} | {hd_A:.2f} | {f1_A:.4f}")
print(f"Test B | {dice_B:.4f} | {hd_B:.2f} | {f1_B:.4f}")
print("------------------------------------------")

In [ ]:
# ============================================================
# VISUALIZACIÓN FINAL DE MÁSCARAS (train o test)
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# ------- CONFIGURACIÓN --------
# Usa "train" o "test"
mode = "train"   # <-- cambiar a "test" si querés ver testA/testB

if mode == "train":
    images_dir = Path(base_path) / "data/train/images_png"
    masks_dir  = Path(base_path) / "data/train/masks_png"
    ext = "*.png"
    mask_suffix = "_anno.png"
else:
    images_dir = Path(base_path) / "data/test/images"
    masks_dir  = Path(base_path) / "data/test/masks"
    ext = "*.bmp"
    mask_suffix = "_anno.bmp"

# ---- cargar lista de imágenes ----
image_list = sorted(images_dir.glob(ext))
print("Número de imágenes encontradas:", len(image_list))

if len(image_list) == 0:
    raise ValueError("❌ No hay imágenes en la carpeta seleccionada.")

# Elegir imagen (0 = primera)
idx = 4
example_path = image_list[idx]

mask_path = masks_dir / f"{example_path.stem}{mask_suffix}"

if not mask_path.exists():
    raise ValueError(f"❌ No se encontró máscara real para: {example_path.name}")

print("Imagen:", example_path.name)
print("Máscara:", mask_path.name)

# ---- cargar imagen original ----
image = np.array(Image.open(example_path).convert("RGB"))
H, W = image.shape[:2]

# ---- cargar GT ----
gt_mask = np.array(Image.open(mask_path).convert("L"))
gt_mask_bin = (gt_mask > 0).astype(np.uint8)

# ---- aplicar val_transforms ----
aug = val_transforms(image=image)
img_tensor = aug["image"].unsqueeze(0).to(device)

# ---- inferencia ----
model.eval()
with torch.no_grad():
    logits = model(img_tensor)
    prob_512 = torch.sigmoid(logits)[0,0].cpu().numpy()

# ---- reescalar a tamaño original ----
prob = cv2.resize(prob_512, (W, H), interpolation=cv2.INTER_LINEAR)

# ---- threshold + clean + watershed ----
pred_bin_raw = (prob > best_thr).astype(np.uint8)
pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
pred_ws      = (labels_ws > 0).astype(np.uint8)

# ---- plot ----
plt.figure(figsize=(18,5))

plt.subplot(1,4,1)
plt.imshow(image)
plt.title("Imagen original")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(pred_clean, cmap="gray")
plt.title("Pred limpia")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(pred_ws, cmap="gray")
plt.title("Pred post-watershed")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(gt_mask_bin, cmap="gray")
plt.title("Máscara real (GT)")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GlandNet - UNet 3-level + Tversky + Watershed + BCE ponderado por bordes
# ============================================================

# Montar Drive

# ---- Instalación de librerías ----
!pip install -q albumentations==1.4.3 scikit-image scipy

# ---- Imports generales ----
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F          # <- para BCE ponderado
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.metrics import hausdorff_distance
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

from tqdm import tqdm

# ============================================================
# 1) Cargar pares imagen-máscara (CSV ya generado)
# ============================================================
base_path = "."
pairs_csv = Path(base_path) / "metadata/train_pairs.csv"

df_pairs = pd.read_csv(pairs_csv)
# Fix: Update paths to use the correct base directory (GlandNet2)
df_pairs["image_path"] = df_pairs["image_path"].str.replace("GlandNet/data", "GlandNet2/data")
df_pairs["mask_path"]  = df_pairs["mask_path"].str.replace("GlandNet/data", "GlandNet2/data")
print(df_pairs.head())
print("Total pares:", len(df_pairs))

# ============================================================
# 2) Split train / val
# ============================================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

indices = list(range(len(df_pairs)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx   = indices[split:]

df_train = df_pairs.iloc[train_idx].reset_index(drop=True)
df_val   = df_pairs.iloc[val_idx].reset_index(drop=True)

print("Train:", len(df_train), "Val:", len(df_val))

# ============================================================
# 3) Dataset y transforms
# ============================================================
class GlasSegmentationDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]        # [3,H,W]
            mask  = augmented["mask"]        # [H,W] o [1,H,W]
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)     # [1,H,W]
        else:
            image = image.transpose(2,0,1).astype("float32") / 255.
            image = torch.tensor(image)
            mask  = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask

# ---- Augmentations ----
train_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            p=0.5, border_mode=0
        ),
        # Morfología
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=10, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        # Color / iluminación
        A.HueSaturationValue(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        # Normalización
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

train_ds = GlasSegmentationDataset(df_train, transforms=train_transforms)
val_ds   = GlasSegmentationDataset(df_val,   transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2)

print("len(train_ds), len(val_ds) =", len(train_ds), len(val_ds))

# ============================================================
# 4) Modelo: UNet 3 niveles
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.pool  = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)

        # Decoder
        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        c3 = self.down3(p2)
        p3 = self.pool(c3)

        bn = self.bottleneck(p3)

        u3 = self.up3(bn)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)

        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)

# ============================================================
# 5) Funciones de pérdida
# ============================================================
def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2,3))
    den = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
    dice = num / den
    return 1 - dice.mean()

def tversky_loss(logits, targets, alpha=0.7, beta=0.3, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=(2,3))
    fp = (probs * (1 - targets)).sum(dim=(2,3))
    fn = ((1 - probs) * targets).sum(dim=(2,3))
    tversky = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
    return 1 - tversky.mean()

def compute_edge_weight_map(masks, w_edge=5.0, w_non_edge=1.0, kernel_size=3):
    """
    masks: tensor [B,1,H,W] con valores 0/1.
    Devuelve mapa de pesos [B,1,H,W] con mayor peso en píxeles de borde.
    """
    # max y min locales en una ventana kxk
    max_pool = F.max_pool2d(masks, kernel_size=kernel_size, stride=1,
                            padding=kernel_size//2)
    min_pool = -F.max_pool2d(-masks, kernel_size=kernel_size, stride=1,
                             padding=kernel_size//2)

    # borde: ventana contiene tanto 0 como 1
    edge = (max_pool - min_pool) > 0  # bool

    weights = torch.full_like(masks, w_non_edge)
    weights[edge] = w_edge
    return weights

# ============================================================
# 6) Setup entrenamiento
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet().to(device)
bce   = nn.BCEWithLogitsLoss(reduction="none")   # usaremos weight map
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8
)

patience = 15
epochs_no_improve = 0
best_val_dice = 0.0
num_epochs = 120

models_dir = Path(base_path) / "models"
models_dir.mkdir(exist_ok=True)
best_model_path = models_dir / "best_unet_glas_3lvl_tversky_edge_bce.pth"

train_losses = []
val_dices    = []

# ============================================================
# 7) Loop de entrenamiento
# ============================================================
for epoch in range(1, num_epochs+1):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)

        # BCE ponderado por bordes
        weight_map = compute_edge_weight_map(masks)          # [B,1,H,W]
        bce_per_px = F.binary_cross_entropy_with_logits(
            logits, masks, weight=weight_map, reduction="none"
        )
        loss_bce = bce_per_px.mean()

        loss_d       = dice_loss(logits, masks)
        loss_tversky = tversky_loss(logits, masks)

        loss = 0.3 * loss_bce + 0.3 * loss_d + 0.4 * loss_tversky

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)
    train_losses.append(train_loss)

    # --- VALID ---
    model.eval()
    val_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            logits = model(imgs)
            d = 1 - dice_loss(logits, masks).item()
            val_dice += d
            n_batches += 1

    val_dice /= max(n_batches, 1)
    val_dices.append(val_dice)

    scheduler.step(val_dice)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_dice={val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_model_path)
        print(f"  ↳ Nuevo mejor modelo guardado en {best_model_path}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  ↳ Sin mejora en val_dice por {epochs_no_improve} epoch(s)")
        if epochs_no_improve >= patience:
            print(f"🔚 Early stopping activado en epoch {epoch}")
            break

# ============================================================
# 8) Curvas de entrenamiento
# ============================================================
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida de entrenamiento")
plt.grid(True)
plt.legend()

plt.subplot(1,2,2)
plt.plot(val_dices, label="Val Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Dice en validación")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 9) Post-proceso y métricas tipo GlaS
# ============================================================
def clean_mask(pred_bin, min_size=30, opening_radius=0):
    mask = pred_bin.astype(bool)
    mask = remove_small_objects(mask, min_size=min_size)
    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)
    return mask.astype(np.uint8)

def split_glands_watershed(pred_bin, footprint_size=15):
    distance = ndi.distance_transform_edt(pred_bin)
    coords = peak_local_max(
        distance,
        footprint=np.ones((footprint_size, footprint_size)),
        labels=pred_bin
    )
    markers_mask = np.zeros(distance.shape, dtype=bool)
    if coords.size > 0:
        markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    labels_ws = watershed(-distance, markers, mask=pred_bin)
    return labels_ws

def compute_dice(mask1, mask2, eps=1e-6):
    inter = np.logical_and(mask1, mask2).sum()
    union = mask1.sum() + mask2.sum()
    return 2 * inter / (union + eps)

def compute_f1(tp, fp, fn):
    return tp / (tp + 0.5*(fp + fn) + 1e-6)

def object_metrics(pred_mask, gt_mask):
    labeled_pred = label(pred_mask)
    labeled_gt   = label(gt_mask)

    pred_regions = regionprops(labeled_pred)
    gt_regions   = regionprops(labeled_gt)

    num_pred = len(pred_regions)
    num_gt   = len(gt_regions)

    object_dices = []
    hausdorffs   = []

    dice_matrix = np.zeros((num_gt, num_pred))

    for i, gt in enumerate(gt_regions):
        gt_mask_i = (labeled_gt == gt.label)
        for j, pr in enumerate(pred_regions):
            pr_mask_j = (labeled_pred == pr.label)
            d = compute_dice(gt_mask_i, pr_mask_j)
            dice_matrix[i, j] = d

    tp = 0
    fn = 0
    matched_pred = set()

    for i in range(num_gt):
        if num_pred == 0:
            fn += 1
            continue
        j = np.argmax(dice_matrix[i])
        best_dice = dice_matrix[i, j]

        gt_mask_i = (labeled_gt == (i+1))
        pr_mask_j = (labeled_pred == (j+1))

        if best_dice > 0.5:
            tp += 1
            matched_pred.add(j)
            object_dices.append(best_dice)
            hd = hausdorff_distance(gt_mask_i, pr_mask_j)
            hausdorffs.append(hd)
        else:
            fn += 1

    fp = num_pred - len(matched_pred)
    f1 = compute_f1(tp, fp, fn)

    mean_dice = np.mean(object_dices) if len(object_dices) > 0 else 0.0
    mean_hd   = np.mean(hausdorffs)   if len(hausdorffs)   > 0 else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "object_dice_mean": mean_dice,
        "hausdorff_mean":   mean_hd,
        "F1": f1
    }

# ============================================================
# 10) Cargar mejor modelo
# ============================================================
model = UNet().to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("Modelo recargado desde:", best_model_path)

# ============================================================
# 11) Barrido de thresholds en validación
# ============================================================
thresholds = np.linspace(0.3, 0.7, 9)
results_by_thr = []

for thr in thresholds:
    all_results_thr = []

    for imgs, gt_mask in val_loader:
        imgs = imgs.to(device)

        with torch.no_grad():
            pred_logits = model(imgs)
            pred_prob   = torch.sigmoid(pred_logits).cpu().numpy()[0,0]

        pred_bin_raw = (pred_prob > thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        gt_mask_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        res = object_metrics(pred_final, gt_mask_np)
        all_results_thr.append(res)

    mean_dice_thr = np.mean([r["object_dice_mean"] for r in all_results_thr])
    mean_hd_thr   = np.mean([r["hausdorff_mean"]   for r in all_results_thr])
    mean_f1_thr   = np.mean([r["F1"]               for r in all_results_thr])

    results_by_thr.append((thr, mean_dice_thr, mean_hd_thr, mean_f1_thr))
    print(f"thr={thr:.2f} | Dice={mean_dice_thr:.4f} | HD={mean_hd_thr:.2f} | F1={mean_f1_thr:.4f}")

best_thr, best_dice, best_hd, best_f1 = max(results_by_thr, key=lambda x: x[3])
print("\n>>> Mejor threshold según F1:", best_thr)
print(f"    Dice={best_dice:.4f} | HD={best_hd:.2f} | F1={best_f1:.4f}")

# ============================================================
# 12) Métricas globales en VALIDATION
# ============================================================
model.eval()
all_results = []

for imgs, gt_mask in tqdm(val_loader, total=len(val_loader)):
    imgs = imgs.to(device)

    with torch.no_grad():
        pred_logits = model(imgs)
        pred_prob   = torch.sigmoid(pred_logits).cpu().numpy()[0,0]

    pred_bin_raw = (pred_prob > best_thr).astype(np.uint8)
    pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
    labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
    pred_final   = (labels_ws > 0).astype(np.uint8)

    gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

    res = object_metrics(pred_final, gt_np)
    all_results.append(res)

mean_object_dice = np.mean([r["object_dice_mean"] for r in all_results])
mean_hausdorff   = np.mean([r["hausdorff_mean"]   for r in all_results])
mean_f1          = np.mean([r["F1"]               for r in all_results])

print("\n===== MÉTRICAS VALIDACIÓN =====")
print(f"Object Dice promedio: {mean_object_dice:.4f}")
print(f"Hausdorff promedio:   {mean_hausdorff:.2f}")
print(f"F1 object-level:      {mean_f1:.4f}")

# ============================================================
# 13) TEST A/B
# ============================================================
test_images_dir = Path(base_path) / "data/test/images"
test_masks_dir  = Path(base_path) / "data/test/masks"

img_files = sorted([p for p in test_images_dir.iterdir()
                    if p.suffix.lower() == ".bmp"])

pairs_test = []
for img_path in img_files:
    name = img_path.stem              # testA_1, testB_2, ...
    mask_path = test_masks_dir / f"{name}_anno.bmp"
    if mask_path.exists():
        pairs_test.append({
            "image_path": str(img_path),
            "mask_path":  str(mask_path)
        })
    else:
        print("⚠ No se encontró máscara para:", img_path.name)

df_test = pd.DataFrame(pairs_test)
print("Total pares en test:", len(df_test))
df_test.head()

df_test["subset"] = df_test["image_path"].apply(
    lambda x: "A" if "testA" in x else "B"
)

df_test_A = df_test[df_test["subset"]=="A"].reset_index(drop=True)
df_test_B = df_test[df_test["subset"]=="B"].reset_index(drop=True)

class GlasSegTestWithGT(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            image = torch.tensor(image.transpose(2,0,1) / 255.)
            mask  = torch.tensor(mask).unsqueeze(0)

        return image, mask

testA_ds = GlasSegTestWithGT(df_test_A, transforms=val_transforms)
testB_ds = GlasSegTestWithGT(df_test_B, transforms=val_transforms)

testA_loader = DataLoader(testA_ds, batch_size=1, shuffle=False)
testB_loader = DataLoader(testB_ds, batch_size=1, shuffle=False)

def evaluate_GlaS_loader(loader, model, device, best_thr):
    results = []
    model.eval()

    for imgs, gt_mask in tqdm(loader, total=len(loader)):
        imgs = imgs.to(device)
        gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        with torch.no_grad():
            pred = torch.sigmoid(model(imgs)).cpu().numpy()[0,0]

        pred_bin_raw = (pred > best_thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        res = object_metrics(pred_final, gt_np)
        results.append(res)

    mean_dice = np.mean([r["object_dice_mean"] for r in results])
    mean_hd   = np.mean([r["hausdorff_mean"]   for r in results])
    mean_f1   = np.mean([r["F1"]               for r in results])

    return mean_dice, mean_hd, mean_f1

print("\n===== MÉTRICAS TEST A =====")
dice_A, hd_A, f1_A = evaluate_GlaS_loader(testA_loader, model, device, best_thr)
print(f"Object Dice (A): {dice_A:.4f}")
print(f"Hausdorff (A):   {hd_A:.2f}")
print(f"F1 objetual (A): {f1_A:.4f}")

print("\n===== MÉTRICAS TEST B =====")
dice_B, hd_B, f1_B = evaluate_GlaS_loader(testB_loader, model, device, best_thr)
print(f"Object Dice (B): {dice_B:.4f}")
print(f"Hausdorff (B):   {hd_B:.2f}")
print(f"F1 objetual (B): {f1_B:.4f}")

print("\n========== RESUMEN FINAL TEST A/B ==========")
print("Subset | Dice | Hausdorff | F1")
print("------------------------------------------")
print(f"Test A | {dice_A:.4f} | {hd_A:.2f} | {f1_A:.4f}")
print(f"Test B | {dice_B:.4f} | {hd_B:.2f} | {f1_B:.4f}")
print("------------------------------------------")

In [ ]:
# ============================================================
# VISUALIZACIÓN FINAL DE MÁSCARAS (train o test)
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# ------- CONFIGURACIÓN --------
# Usa "train" o "test"
mode = "train"   # <-- cambiar a "test" si querés ver testA/testB

if mode == "train":
    images_dir = Path(base_path) / "data/train/images_png"
    masks_dir  = Path(base_path) / "data/train/masks_png"
    ext = "*.png"
    mask_suffix = "_anno.png"
else:
    images_dir = Path(base_path) / "data/test/images"
    masks_dir  = Path(base_path) / "data/test/masks"
    ext = "*.bmp"
    mask_suffix = "_anno.bmp"

# ---- cargar lista de imágenes ----
image_list = sorted(images_dir.glob(ext))
print("Número de imágenes encontradas:", len(image_list))

if len(image_list) == 0:
    raise ValueError("❌ No hay imágenes en la carpeta seleccionada.")

# Elegir imagen (0 = primera)
idx = 0
example_path = image_list[idx]

mask_path = masks_dir / f"{example_path.stem}{mask_suffix}"

if not mask_path.exists():
    raise ValueError(f"❌ No se encontró máscara real para: {example_path.name}")

print("Imagen:", example_path.name)
print("Máscara:", mask_path.name)

# ---- cargar imagen original ----
image = np.array(Image.open(example_path).convert("RGB"))
H, W = image.shape[:2]

# ---- cargar GT ----
gt_mask = np.array(Image.open(mask_path).convert("L"))
gt_mask_bin = (gt_mask > 0).astype(np.uint8)

# ---- aplicar val_transforms ----
aug = val_transforms(image=image)
img_tensor = aug["image"].unsqueeze(0).to(device)

# ---- inferencia ----
model.eval()
with torch.no_grad():
    logits = model(img_tensor)
    prob_512 = torch.sigmoid(logits)[0,0].cpu().numpy()

# ---- reescalar a tamaño original ----
prob = cv2.resize(prob_512, (W, H), interpolation=cv2.INTER_LINEAR)

# ---- threshold + clean + watershed ----
pred_bin_raw = (prob > best_thr).astype(np.uint8)
pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
pred_ws      = (labels_ws > 0).astype(np.uint8)

# ---- plot ----
plt.figure(figsize=(18,5))

plt.subplot(1,4,1)
plt.imshow(image)
plt.title("Imagen original")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(pred_clean, cmap="gray")
plt.title("Pred limpia")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(pred_ws, cmap="gray")
plt.title("Pred post-watershed")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(gt_mask_bin, cmap="gray")
plt.title("Máscara real (GT)")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GlandNet v2 - UNet 4-level + multi-canal (interior + borde)
# ============================================================

# Montar Drive

# ---- Instalación de librerías ----
!pip install -q albumentations==1.4.3 scikit-image scipy

# ---- Imports generales ----
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.metrics import hausdorff_distance
from scipy import ndimage as ndi
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

from tqdm import tqdm

# ============================================================
# 1) Cargar pares imagen-máscara (CSV ya generado)
# ============================================================
base_path = "."
pairs_csv = Path(base_path) / "metadata/train_pairs.csv"

df_pairs = pd.read_csv(pairs_csv)

# Fix paths si el CSV venía de GlandNet anterior
df_pairs["image_path"] = df_pairs["image_path"].str.replace("GlandNet/data", "GlandNet2/data")
df_pairs["mask_path"]  = df_pairs["mask_path"].str.replace("GlandNet/data", "GlandNet2/data")

print(df_pairs.head())
print("Total pares:", len(df_pairs))

# ============================================================
# 2) Split train / val
# ============================================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

indices = list(range(len(df_pairs)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx   = indices[split:]

df_train = df_pairs.iloc[train_idx].reset_index(drop=True)
df_val   = df_pairs.iloc[val_idx].reset_index(drop=True)

print("Train:", len(df_train), "Val:", len(df_val))

# ============================================================
# 3) Dataset y transforms (multi-canal interior + borde)
# ============================================================
def build_border(mask_bin, iterations=1):
    """
    mask_bin: np.array [H,W] binaria (0/1)
    Devuelve borde morfológico (0/1)
    """
    dil = binary_dilation(mask_bin, iterations=iterations)
    ero = binary_erosion(mask_bin, iterations=iterations)
    border = np.logical_xor(dil, ero).astype("float32")
    return border

class GlasSegmentationDataset(Dataset):
    """
    Devuelve:
      image: tensor [3,H,W]
      mask:  tensor [2,H,W]
        canal 0 -> interior
        canal 1 -> borde
    """
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask_bin = (mask > 0).astype("float32")

        interior = mask_bin
        border   = build_border(mask_bin, iterations=1)

        # HWC → (H, W, 2)
        mask_multi = np.stack([interior, border], axis=-1)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask_multi)
            image = augmented["image"]      # tensor [3,H,W]
            mask  = augmented["mask"]       # HWC → hay que permutar

            if mask.ndim == 3:              # [H,W,2] → [2,H,W]
                mask = mask.permute(2, 0, 1)

        else:
            image = torch.tensor(image.transpose(2,0,1)/255.0)
            mask  = torch.tensor(mask_multi.transpose(2,0,1))

        return image.float(), mask.float()


# ---- Augmentations ----
train_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            p=0.5, border_mode=0
        ),
        # Morfología / deformaciones suaves
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=10, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        # Color / iluminación
        A.HueSaturationValue(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        # Normalización
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

train_ds = GlasSegmentationDataset(df_train, transforms=train_transforms)
val_ds   = GlasSegmentationDataset(df_val,   transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2)

print("len(train_ds), len(val_ds) =", len(train_ds), len(val_ds))

# ============================================================
# 4) Modelo: UNet 4 niveles (64-128-256-512-1024), out_ch = 2
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet4L(nn.Module):
    def __init__(self, in_ch=3, out_ch=2):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool  = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4   = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        c3 = self.down3(p2)
        p3 = self.pool(c3)

        c4 = self.down4(p3)
        p4 = self.pool(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        u4 = torch.cat([u4, c4], dim=1)
        c4 = self.conv4(u4)

        u3 = self.up3(c4)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)

        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)   # [B,2,H,W]

# ============================================================
# 5) Funciones de pérdida (multi-canal)
# ============================================================
def dice_loss(logits, targets, eps=1e-6):
    """
    logits/targets: [B,2,H,W]
    """
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2,3))          # [B,2]
    den = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
    dice = num / den                                    # [B,2]
    return 1 - dice.mean()

def tversky_loss(logits, targets, alpha=0.7, beta=0.3, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=(2,3))
    fp = (probs * (1 - targets)).sum(dim=(2,3))
    fn = ((1 - probs) * targets).sum(dim=(2,3))
    tversky = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
    return 1 - tversky.mean()

def bce_multi_with_border_weight(logits, targets, w_border=3.0):
    """
    BCE por canal, con más peso en canal 1 (borde).
    logits/targets: [B,2,H,W]
    """
    bce_per_px = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )   # [B,2,H,W]

    weights = torch.ones_like(targets)
    # canal 1 = borde
    weights[:,1:2,:,:] = w_border

    loss = (bce_per_px * weights).mean()
    return loss

# ============================================================
# 6) Setup entrenamiento
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet4L().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8
)

patience = 15
epochs_no_improve = 0
best_val_dice = 0.0
num_epochs = 120

models_dir = Path(base_path) / "models"
models_dir.mkdir(exist_ok=True)
best_model_path = models_dir / "best_unet4L_multichannel.pth"

train_losses = []
val_dices    = []

# ============================================================
# 7) Loop de entrenamiento
# ============================================================
for epoch in range(1, num_epochs+1):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)          # [B,2,H,W]

        logits = model(imgs)              # [B,2,H,W]

        loss_bce     = bce_multi_with_border_weight(logits, masks, w_border=3.0)
        loss_d       = dice_loss(logits, masks)
        loss_tversky = tversky_loss(logits, masks)

        loss = 0.3 * loss_bce + 0.3 * loss_d + 0.4 * loss_tversky

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)
    train_losses.append(train_loss)

    # --- VALID ---
    model.eval()
    val_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)      # [B,2,H,W]
            logits = model(imgs)
            d = 1 - dice_loss(logits, masks).item()
            val_dice += d
            n_batches += 1

    val_dice /= max(n_batches, 1)
    val_dices.append(val_dice)

    scheduler.step(val_dice)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_dice={val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_model_path)
        print(f"  ↳ Nuevo mejor modelo guardado en {best_model_path}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  ↳ Sin mejora en val_dice por {epochs_no_improve} epoch(s)")
        if epochs_no_improve >= patience:
            print(f"🔚 Early stopping activado en epoch {epoch}")
            break

# ============================================================
# 8) Curvas de entrenamiento
# ============================================================
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida de entrenamiento")
plt.grid(True)
plt.legend()

plt.subplot(1,2,2)
plt.plot(val_dices, label="Val Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Dice en validación")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 9) Post-proceso y métricas tipo GlaS
# ============================================================
def clean_mask(pred_bin, min_size=30, opening_radius=0):
    mask = pred_bin.astype(bool)
    mask = remove_small_objects(mask, min_size=min_size)
    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)
    return mask.astype(np.uint8)

def split_glands_watershed(pred_bin, footprint_size=15):
    distance = ndi.distance_transform_edt(pred_bin)
    coords = peak_local_max(
        distance,
        footprint=np.ones((footprint_size, footprint_size)),
        labels=pred_bin
    )
    markers_mask = np.zeros(distance.shape, dtype=bool)
    if coords.size > 0:
        markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    labels_ws = watershed(-distance, markers, mask=pred_bin)
    return labels_ws

def compute_dice(mask1, mask2, eps=1e-6):
    inter = np.logical_and(mask1, mask2).sum()
    union = mask1.sum() + mask2.sum()
    return 2 * inter / (union + eps)

def compute_f1(tp, fp, fn):
    return tp / (tp + 0.5*(fp + fn) + 1e-6)

def object_metrics(pred_mask, gt_mask):
    labeled_pred = label(pred_mask)
    labeled_gt   = label(gt_mask)

    pred_regions = regionprops(labeled_pred)
    gt_regions   = regionprops(labeled_gt)

    num_pred = len(pred_regions)
    num_gt   = len(gt_regions)

    object_dices = []
    hausdorffs   = []

    dice_matrix = np.zeros((num_gt, num_pred))

    for i, gt in enumerate(gt_regions):
        gt_mask_i = (labeled_gt == gt.label)
        for j, pr in enumerate(pred_regions):
            pr_mask_j = (labeled_pred == pr.label)
            d = compute_dice(gt_mask_i, pr_mask_j)
            dice_matrix[i, j] = d

    tp = 0
    fn = 0
    matched_pred = set()

    for i in range(num_gt):
        if num_pred == 0:
            fn += 1
            continue
        j = np.argmax(dice_matrix[i])
        best_dice = dice_matrix[i, j]

        gt_mask_i = (labeled_gt == (i+1))
        pr_mask_j = (labeled_pred == (j+1))

        if best_dice > 0.5:
            tp += 1
            matched_pred.add(j)
            object_dices.append(best_dice)
            hd = hausdorff_distance(gt_mask_i, pr_mask_j)
            hausdorffs.append(hd)
        else:
            fn += 1

    fp = num_pred - len(matched_pred)
    f1 = compute_f1(tp, fp, fn)

    mean_dice = np.mean(object_dices) if len(object_dices) > 0 else 0.0
    mean_hd   = np.mean(hausdorffs)   if len(hausdorffs)   > 0 else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "object_dice_mean": mean_dice,
        "hausdorff_mean":   mean_hd,
        "F1": f1
    }

# ============================================================
# 10) Cargar mejor modelo
# ============================================================
model = UNet4L().to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("Modelo recargado desde:", best_model_path)

# ============================================================
# 11) Barrido de thresholds en validación
# ============================================================
thresholds = np.linspace(0.3, 0.7, 9)
results_by_thr = []

for thr in thresholds:
    all_results_thr = []

    for imgs, gt_mask in val_loader:
        imgs = imgs.to(device)          # gt_mask: [B,2,H,W] (canal 0 = interior)

        with torch.no_grad():
            pred_logits = model(imgs)   # [B,2,H,W]
            prob_2ch    = torch.sigmoid(pred_logits).cpu().numpy()[0]   # [2,H,W]

        # Unión de interior + borde como probabilidad de glándula
        gland_prob = np.max(prob_2ch, axis=0)       # [H,W]

        pred_bin_raw = (gland_prob > thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        # GT: usamos canal 0 (interior original)
        gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        res = object_metrics(pred_final, gt_np)
        all_results_thr.append(res)

    mean_dice_thr = np.mean([r["object_dice_mean"] for r in all_results_thr])
    mean_hd_thr   = np.mean([r["hausdorff_mean"]   for r in all_results_thr])
    mean_f1_thr   = np.mean([r["F1"]               for r in all_results_thr])

    results_by_thr.append((thr, mean_dice_thr, mean_hd_thr, mean_f1_thr))
    print(f"thr={thr:.2f} | Dice={mean_dice_thr:.4f} | HD={mean_hd_thr:.2f} | F1={mean_f1_thr:.4f}")

best_thr, best_dice, best_hd, best_f1 = max(results_by_thr, key=lambda x: x[3])
print("\n>>> Mejor threshold según F1:", best_thr)
print(f"    Dice={best_dice:.4f} | HD={best_hd:.2f} | F1={best_f1:.4f}")

# ============================================================
# 12) Métricas globales en VALIDATION
# ============================================================
model.eval()
all_results = []

for imgs, gt_mask in tqdm(val_loader, total=len(val_loader)):
    imgs = imgs.to(device)

    with torch.no_grad():
        pred_logits = model(imgs)
        prob_2ch    = torch.sigmoid(pred_logits).cpu().numpy()[0]

    gland_prob   = np.max(prob_2ch, axis=0)
    pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
    pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
    labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
    pred_final   = (labels_ws > 0).astype(np.uint8)

    gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

    res = object_metrics(pred_final, gt_np)
    all_results.append(res)

mean_object_dice = np.mean([r["object_dice_mean"] for r in all_results])
mean_hausdorff   = np.mean([r["hausdorff_mean"]   for r in all_results])
mean_f1          = np.mean([r["F1"]               for r in all_results])

print("\n===== MÉTRICAS VALIDACIÓN =====")
print(f"Object Dice promedio: {mean_object_dice:.4f}")
print(f"Hausdorff promedio:   {mean_hausdorff:.2f}")
print(f"F1 object-level:      {mean_f1:.4f}")

# ============================================================
# 13) TEST A/B
# ============================================================
test_images_dir = Path(base_path) / "data/test/images"
test_masks_dir  = Path(base_path) / "data/test/masks"

img_files = sorted([p for p in test_images_dir.iterdir()
                    if p.suffix.lower() == ".bmp"])

pairs_test = []
for img_path in img_files:
    name = img_path.stem              # testA_1, testB_2, ...
    mask_path = test_masks_dir / f"{name}_anno.bmp"
    if mask_path.exists():
        pairs_test.append({
            "image_path": str(img_path),
            "mask_path":  str(mask_path)
        })
    else:
        print("⚠ No se encontró máscara para:", img_path.name)

df_test = pd.DataFrame(pairs_test)
print("Total pares en test:", len(df_test))
df_test.head()

df_test["subset"] = df_test["image_path"].apply(
    lambda x: "A" if "testA" in x else "B"
)

df_test_A = df_test[df_test["subset"]=="A"].reset_index(drop=True)
df_test_B = df_test[df_test["subset"]=="B"].reset_index(drop=True)

class GlasSegTestWithGT(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask_bin = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask_bin)
            image = augmented["image"]               # [3,H,W]
            mask  = augmented["mask"].unsqueeze(0)   # [1,H,W]
        else:
            image = torch.tensor(image.transpose(2,0,1) / 255.)
            mask  = torch.tensor(mask_bin).unsqueeze(0)

        return image.float(), mask.float()

testA_ds = GlasSegTestWithGT(df_test_A, transforms=val_transforms)
testB_ds = GlasSegTestWithGT(df_test_B, transforms=val_transforms)

testA_loader = DataLoader(testA_ds, batch_size=1, shuffle=False)
testB_loader = DataLoader(testB_ds, batch_size=1, shuffle=False)

def evaluate_GlaS_loader(loader, model, device, best_thr):
    results = []
    model.eval()

    for imgs, gt_mask in tqdm(loader, total=len(loader)):
        imgs = imgs.to(device)
        gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        with torch.no_grad():
            logits = model(imgs)                          # [1,2,H,W]
            prob_2ch = torch.sigmoid(logits).cpu().numpy()[0]

        gland_prob   = np.max(prob_2ch, axis=0)
        pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        res = object_metrics(pred_final, gt_np)
        results.append(res)

    mean_dice = np.mean([r["object_dice_mean"] for r in results])
    mean_hd   = np.mean([r["hausdorff_mean"]   for r in results])
    mean_f1   = np.mean([r["F1"]               for r in results])

    return mean_dice, mean_hd, mean_f1

print("\n===== MÉTRICAS TEST A =====")
dice_A, hd_A, f1_A = evaluate_GlaS_loader(testA_loader, model, device, best_thr)
print(f"Object Dice (A): {dice_A:.4f}")
print(f"Hausdorff (A):   {hd_A:.2f}")
print(f"F1 objetual (A): {f1_A:.4f}")

print("\n===== MÉTRICAS TEST B =====")
dice_B, hd_B, f1_B = evaluate_GlaS_loader(testB_loader, model, device, best_thr)
print(f"Object Dice (B): {dice_B:.4f}")
print(f"Hausdorff (B):   {hd_B:.2f}")
print(f"F1 objetual (B): {f1_B:.4f}")

print("\n========== RESUMEN FINAL TEST A/B ==========")
print("Subset | Dice | Hausdorff | F1")
print("------------------------------------------")
print(f"Test A | {dice_A:.4f} | {hd_A:.2f} | {f1_A:.4f}")
print(f"Test B | {dice_B:.4f} | {hd_B:.2f} | {f1_B:.4f}")
print("------------------------------------------")

In [ ]:
# ============================================================
# Visualización de ejemplo (train o test)
# ============================================================
import cv2

mode = "train"   # "train" o "test"

if mode == "train":
    images_dir = Path(base_path) / "data/train/images_png"
    masks_dir  = Path(base_path) / "data/train/masks_png"
    ext = "*.png"
    mask_suffix = "_anno.png"
else:
    images_dir = Path(base_path) / "data/test/images"
    masks_dir  = Path(base_path) / "data/test/masks"
    ext = "*.bmp"
    mask_suffix = "_anno.bmp"

image_list = sorted(images_dir.glob(ext))
print("Número de imágenes encontradas:", len(image_list))

idx = 30  # cambiá el índice para ver otras
example_path = image_list[idx]
mask_path = masks_dir / f"{example_path.stem}{mask_suffix}"

print("Imagen:", example_path.name)
print("Máscara:", mask_path.name)

image = np.array(Image.open(example_path).convert("RGB"))
H, W = image.shape[:2]

gt_mask = np.array(Image.open(mask_path).convert("L"))
gt_mask_bin = (gt_mask > 0).astype(np.uint8)

aug = val_transforms(image=image)
img_tensor = aug["image"].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    logits = model(img_tensor)                     # [1,2,512,512]
    prob_2ch_512 = torch.sigmoid(logits)[0].cpu().numpy()  # [2,512,512]

# Unión interior + borde
gland_prob_512 = np.max(prob_2ch_512, axis=0)     # [512,512]

# Reescalar a tamaño original
gland_prob = cv2.resize(gland_prob_512, (W, H), interpolation=cv2.INTER_LINEAR)

pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
pred_ws      = (labels_ws > 0).astype(np.uint8)

plt.figure(figsize=(18,5))
plt.subplot(1,4,1); plt.imshow(image);       plt.title("Imagen original");      plt.axis("off")
plt.subplot(1,4,2); plt.imshow(pred_clean, cmap="gray"); plt.title("Pred limpia"); plt.axis("off")
plt.subplot(1,4,3); plt.imshow(pred_ws,    cmap="gray"); plt.title("Pred post-watershed"); plt.axis("off")
plt.subplot(1,4,4); plt.imshow(gt_mask_bin, cmap="gray"); plt.title("Máscara real (GT)"); plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GlandNet v2 - UNet 4-level + multi-canal (interior + borde)
# ============================================================

# Montar Drive

# ---- Instalación de librerías ----
!pip install -q albumentations==1.4.3 scikit-image scipy

# ---- Imports generales ----
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.metrics import hausdorff_distance
from scipy import ndimage as ndi
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

from tqdm import tqdm

# ============================================================
# 1) Cargar pares imagen-máscara (CSV ya generado)
# ============================================================
base_path = "."
pairs_csv = Path(base_path) / "metadata/train_pairs.csv"

df_pairs = pd.read_csv(pairs_csv)

# Fix paths si el CSV venía de GlandNet anterior
df_pairs["image_path"] = df_pairs["image_path"].str.replace("GlandNet/data", "GlandNet2/data")
df_pairs["mask_path"]  = df_pairs["mask_path"].str.replace("GlandNet/data", "GlandNet2/data")

print(df_pairs.head())
print("Total pares:", len(df_pairs))

# ============================================================
# 2) Split train / val
# ============================================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

indices = list(range(len(df_pairs)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx   = indices[split:]

df_train = df_pairs.iloc[train_idx].reset_index(drop=True)
df_val   = df_pairs.iloc[val_idx].reset_index(drop=True)

print("Train:", len(df_train), "Val:", len(df_val))

# ============================================================
# 3) Dataset y transforms (multi-canal interior + borde)
# ============================================================
def build_border(mask_bin, iterations=2):
    """
    mask_bin: np.array [H,W] binaria (0/1)
    Devuelve borde morfológico (0/1).
    iterations=2 -> borde más grueso (≈3–5 px)
    """
    dil = binary_dilation(mask_bin, iterations=iterations)
    ero = binary_erosion(mask_bin, iterations=iterations)
    border = np.logical_xor(dil, ero).astype("float32")
    return border

class GlasSegmentationDataset(Dataset):
    """
    Devuelve:
      image: tensor [3,H,W]
      mask:  tensor [2,H,W]
        canal 0 -> interior
        canal 1 -> borde
    """
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask_bin = (mask > 0).astype("float32")

        interior = mask_bin
        # 👇 borde más grueso
        border   = build_border(mask_bin, iterations=2)

        # HWC → (H, W, 2)
        mask_multi = np.stack([interior, border], axis=-1)

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask_multi)
            image = augmented["image"]      # tensor [3,H,W]
            mask  = augmented["mask"]       # [H,W,2] → hay que permutar

            if mask.ndim == 3:              # [H,W,2] → [2,H,W]
                mask = mask.permute(2, 0, 1)

        else:
            image = torch.tensor(image.transpose(2,0,1)/255.0)
            mask  = torch.tensor(mask_multi.transpose(2,0,1))

        return image.float(), mask.float()


# ---- Augmentations ----
train_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            p=0.5, border_mode=0
        ),
        # Morfología / deformaciones suaves
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=10, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        # Color / iluminación
        A.HueSaturationValue(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        # Normalización
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

train_ds = GlasSegmentationDataset(df_train, transforms=train_transforms)
val_ds   = GlasSegmentationDataset(df_val,   transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2)

print("len(train_ds), len(val_ds) =", len(train_ds), len(val_ds))

# ============================================================
# 4) Modelo: UNet 4 niveles (64-128-256-512-1024), out_ch = 2
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet4L(nn.Module):
    def __init__(self, in_ch=3, out_ch=2):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool  = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4   = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        c3 = self.down3(p2)
        p3 = self.pool(c3)

        c4 = self.down4(p3)
        p4 = self.pool(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        u4 = torch.cat([u4, c4], dim=1)
        c4 = self.conv4(u4)

        u3 = self.up3(c4)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)

        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)   # [B,2,H,W]

# ============================================================
# 5) Funciones de pérdida (multi-canal)
# ============================================================
def dice_loss(logits, targets, eps=1e-6):
    """
    logits/targets: [B,2,H,W]
    """
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2,3))          # [B,2]
    den = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
    dice = num / den                                    # [B,2]
    return 1 - dice.mean()

def tversky_loss(logits, targets, alpha=0.7, beta=0.3, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=(2,3))
    fp = (probs * (1 - targets)).sum(dim=(2,3))
    fn = ((1 - probs) * targets).sum(dim=(2,3))
    tversky = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
    return 1 - tversky.mean()

def bce_multi_with_border_weight(logits, targets, w_border=6.0):
    """
    BCE por canal, con más peso en canal 1 (borde).
    logits/targets: [B,2,H,W]
    """
    bce_per_px = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )   # [B,2,H,W]

    weights = torch.ones_like(targets)
    # canal 1 = borde (más peso)
    weights[:,1:2,:,:] = w_border

    loss = (bce_per_px * weights).mean()
    return loss

# ============================================================
# 6) Setup entrenamiento
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet4L().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8
)

patience = 15
epochs_no_improve = 0
best_val_dice = 0.0
num_epochs = 120

models_dir = Path(base_path) / "models"
models_dir.mkdir(exist_ok=True)
best_model_path = models_dir / "best_unet4L_multichannel.pth"

train_losses = []
val_dices    = []

# ============================================================
# 7) Loop de entrenamiento
# ============================================================
for epoch in range(1, num_epochs+1):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)          # [B,2,H,W]

        logits = model(imgs)              # [B,2,H,W]

        # 👇 BCE ponderada (borde más importante)
        loss_bce     = bce_multi_with_border_weight(logits, masks, w_border=6.0)
        loss_d       = dice_loss(logits, masks)
        loss_tversky = tversky_loss(logits, masks)

        loss = 0.3 * loss_bce + 0.3 * loss_d + 0.4 * loss_tversky

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)
    train_losses.append(train_loss)

    # --- VALID ---
    model.eval()
    val_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)      # [B,2,H,W]
            logits = model(imgs)
            d = 1 - dice_loss(logits, masks).item()
            val_dice += d
            n_batches += 1

    val_dice /= max(n_batches, 1)
    val_dices.append(val_dice)

    scheduler.step(val_dice)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_dice={val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_model_path)
        print(f"  ↳ Nuevo mejor modelo guardado en {best_model_path}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  ↳ Sin mejora en val_dice por {epochs_no_improve} epoch(s)")
        if epochs_no_improve >= patience:
            print(f"🔚 Early stopping activado en epoch {epoch}")
            break

# ============================================================
# 8) Curvas de entrenamiento
# ============================================================
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida de entrenamiento")
plt.grid(True)
plt.legend()

plt.subplot(1,2,2)
plt.plot(val_dices, label="Val Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Dice en validación")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 9) Post-proceso y métricas tipo GlaS
# ============================================================
def clean_mask(pred_bin, min_size=30, opening_radius=0):
    mask = pred_bin.astype(bool)
    mask = remove_small_objects(mask, min_size=min_size)
    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)
    return mask.astype(np.uint8)

def split_glands_watershed(pred_bin, footprint_size=25):
    """
    Watershed para separar glándulas pegadas.
    footprint_size=25 -> vecindario más grande para los picos.
    """
    distance = ndi.distance_transform_edt(pred_bin)
    coords = peak_local_max(
        distance,
        footprint=np.ones((footprint_size, footprint_size)),
        labels=pred_bin
    )
    markers_mask = np.zeros(distance.shape, dtype=bool)
    if coords.size > 0:
        markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    labels_ws = watershed(-distance, markers, mask=pred_bin)
    return labels_ws

def compute_dice(mask1, mask2, eps=1e-6):
    inter = np.logical_and(mask1, mask2).sum()
    union = mask1.sum() + mask2.sum()
    return 2 * inter / (union + eps)

def compute_f1(tp, fp, fn):
    return tp / (tp + 0.5*(fp + fn) + 1e-6)

def object_metrics(pred_mask, gt_mask):
    labeled_pred = label(pred_mask)
    labeled_gt   = label(gt_mask)

    pred_regions = regionprops(labeled_pred)
    gt_regions   = regionprops(labeled_gt)

    num_pred = len(pred_regions)
    num_gt   = len(gt_regions)

    object_dices = []
    hausdorffs   = []

    dice_matrix = np.zeros((num_gt, num_pred))

    for i, gt in enumerate(gt_regions):
        gt_mask_i = (labeled_gt == gt.label)
        for j, pr in enumerate(pred_regions):
            pr_mask_j = (labeled_pred == pr.label)
            d = compute_dice(gt_mask_i, pr_mask_j)
            dice_matrix[i, j] = d

    tp = 0
    fn = 0
    matched_pred = set()

    for i in range(num_gt):
        if num_pred == 0:
            fn += 1
            continue
        j = np.argmax(dice_matrix[i])
        best_dice = dice_matrix[i, j]

        gt_mask_i = (labeled_gt == (i+1))
        pr_mask_j = (labeled_pred == (j+1))

        if best_dice > 0.5:
            tp += 1
            matched_pred.add(j)
            object_dices.append(best_dice)
            hd = hausdorff_distance(gt_mask_i, pr_mask_j)
            hausdorffs.append(hd)
        else:
            fn += 1

    fp = num_pred - len(matched_pred)
    f1 = compute_f1(tp, fp, fn)

    mean_dice = np.mean(object_dices) if len(object_dices) > 0 else 0.0
    mean_hd   = np.mean(hausdorffs)   if len(hausdorffs)   > 0 else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "object_dice_mean": mean_dice,
        "hausdorff_mean":   mean_hd,
        "F1": f1
    }

# ============================================================
# 10) Cargar mejor modelo
# ============================================================
model = UNet4L().to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("Modelo recargado desde:", best_model_path)

# ============================================================
# 11) Barrido de thresholds en validación
# ============================================================
thresholds = np.linspace(0.3, 0.7, 9)
results_by_thr = []

for thr in thresholds:
    all_results_thr = []

    for imgs, gt_mask in val_loader:
        imgs = imgs.to(device)          # gt_mask: [B,2,H,W] (canal 0 = interior)

        with torch.no_grad():
            pred_logits = model(imgs)   # [B,2,H,W]
            prob_2ch    = torch.sigmoid(pred_logits).cpu().numpy()[0]   # [2,H,W]

        # Unión de interior + borde como probabilidad de glándula
        gland_prob = np.max(prob_2ch, axis=0)       # [H,W]

        pred_bin_raw = (gland_prob > thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=25)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        # GT: usamos canal 0 (interior original)
        gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        res = object_metrics(pred_final, gt_np)
        all_results_thr.append(res)

    mean_dice_thr = np.mean([r["object_dice_mean"] for r in all_results_thr])
    mean_hd_thr   = np.mean([r["hausdorff_mean"]   for r in all_results_thr])
    mean_f1_thr   = np.mean([r["F1"]               for r in all_results_thr])

    results_by_thr.append((thr, mean_dice_thr, mean_hd_thr, mean_f1_thr))
    print(f"thr={thr:.2f} | Dice={mean_dice_thr:.4f} | HD={mean_hd_thr:.2f} | F1={mean_f1_thr:.4f}")

best_thr, best_dice, best_hd, best_f1 = max(results_by_thr, key=lambda x: x[3])
print("\n>>> Mejor threshold según F1:", best_thr)
print(f"    Dice={best_dice:.4f} | HD={best_hd:.2f} | F1={best_f1:.4f}")

# ============================================================
# 12) Métricas globales en VALIDATION
# ============================================================
model.eval()
all_results = []

for imgs, gt_mask in tqdm(val_loader, total=len(val_loader)):
    imgs = imgs.to(device)

    with torch.no_grad():
        pred_logits = model(imgs)
        prob_2ch    = torch.sigmoid(pred_logits).cpu().numpy()[0]

    gland_prob   = np.max(prob_2ch, axis=0)
    pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
    pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
    labels_ws    = split_glands_watershed(pred_clean, footprint_size=25)
    pred_final   = (labels_ws > 0).astype(np.uint8)

    gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

    res = object_metrics(pred_final, gt_np)
    all_results.append(res)

mean_object_dice = np.mean([r["object_dice_mean"] for r in all_results])
mean_hausdorff   = np.mean([r["hausdorff_mean"]   for r in all_results])
mean_f1          = np.mean([r["F1"]               for r in all_results])

print("\n===== MÉTRICAS VALIDACIÓN ====")
print(f"Object Dice promedio: {mean_object_dice:.4f}")
print(f"Hausdorff promedio:   {mean_hausdorff:.2f}")
print(f"F1 object-level:      {mean_f1:.4f}")

# ============================================================
# 13) TEST A/B
# ============================================================
test_images_dir = Path(base_path) / "data/test/images"
test_masks_dir  = Path(base_path) / "data/test/masks"

img_files = sorted([p for p in test_images_dir.iterdir()
                    if p.suffix.lower() == ".bmp"])

pairs_test = []
for img_path in img_files:
    name = img_path.stem              # testA_1, testB_2, ...
    mask_path = test_masks_dir / f"{name}_anno.bmp"
    if mask_path.exists():
        pairs_test.append({
            "image_path": str(img_path),
            "mask_path":  str(mask_path)
        })
    else:
        print("⚠ No se encontró máscara para:", img_path.name)

df_test = pd.DataFrame(pairs_test)
print("Total pares en test:", len(df_test))
df_test.head()

df_test["subset"] = df_test["image_path"].apply(
    lambda x: "A" if "testA" in x else "B"
)

df_test_A = df_test[df_test["subset"]=="A"].reset_index(drop=True)
df_test_B = df_test[df_test["subset"]=="B"].reset_index(drop=True)

class GlasSegTestWithGT(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask_bin = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask_bin)
            image = augmented["image"]               # [3,H,W]
            mask  = augmented["mask"].unsqueeze(0)   # [1,H,W]
        else:
            image = torch.tensor(image.transpose(2,0,1) / 255.0)
            mask  = torch.tensor(mask_bin).unsqueeze(0)

        return image.float(), mask.float()

testA_ds = GlasSegTestWithGT(df_test_A, transforms=val_transforms)
testB_ds = GlasSegTestWithGT(df_test_B, transforms=val_transforms)

testA_loader = DataLoader(testA_ds, batch_size=1, shuffle=False)
testB_loader = DataLoader(testB_ds, batch_size=1, shuffle=False)

def evaluate_GlaS_loader(loader, model, device, best_thr):
    results = []
    model.eval()

    for imgs, gt_mask in tqdm(loader, total=len(loader)):
        imgs = imgs.to(device)
        gt_np = gt_mask.cpu().numpy()[0,0].astype(np.uint8)

        with torch.no_grad():
            logits = model(imgs)                          # [1,2,H,W]
            prob_2ch = torch.sigmoid(logits).cpu().numpy()[0]

        gland_prob   = np.max(prob_2ch, axis=0)
        pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
        pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
        labels_ws    = split_glands_watershed(pred_clean, footprint_size=25)
        pred_final   = (labels_ws > 0).astype(np.uint8)

        res = object_metrics(pred_final, gt_np)
        results.append(res)

    mean_dice = np.mean([r["object_dice_mean"] for r in results])
    mean_hd   = np.mean([r["hausdorff_mean"]   for r in results])
    mean_f1   = np.mean([r["F1"]               for r in results])

    return mean_dice, mean_hd, mean_f1

print("\n===== MÉTRICAS TEST A =====")
dice_A, hd_A, f1_A = evaluate_GlaS_loader(testA_loader, model, device, best_thr)
print(f"Object Dice (A): {dice_A:.4f}")
print(f"Hausdorff (A):   {hd_A:.2f}")
print(f"F1 objetual (A): {f1_A:.4f}")

print("\n===== MÉTRICAS TEST B =====")
dice_B, hd_B, f1_B = evaluate_GlaS_loader(testB_loader, model, device, best_thr)
print(f"Object Dice (B): {dice_B:.4f}")
print(f"Hausdorff (B):   {hd_B:.2f}")
print(f"F1 objetual (B): {f1_B:.4f}")

print("\n========== RESUMEN FINAL TEST A/B ==========")
print("Subset | Dice | Hausdorff | F1")
print("------------------------------------------")
print(f"Test A | {dice_A:.4f} | {hd_A:.2f} | {f1_A:.4f}")
print(f"Test B | {dice_B:.4f} | {hd_B:.2f} | {f1_B:.4f}")
print("------------------------------------------")

In [ ]:
# ============================================================
# Visualización de ejemplo (train o test)
# ============================================================
import cv2

mode = "train"   # "train" o "test"

if mode == "train":
    images_dir = Path(base_path) / "data/train/images_png"
    masks_dir  = Path(base_path) / "data/train/masks_png"
    ext = "*.png"
    mask_suffix = "_anno.png"
else:
    images_dir = Path(base_path) / "data/test/images"
    masks_dir  = Path(base_path) / "data/test/masks"
    ext = "*.bmp"
    mask_suffix = "_anno.bmp"

image_list = sorted(images_dir.glob(ext))
print("Número de imágenes encontradas:", len(image_list))

idx = 30  # cambiá el índice para ver otras
example_path = image_list[idx]
mask_path = masks_dir / f"{example_path.stem}{mask_suffix}"

print("Imagen:", example_path.name)
print("Máscara:", mask_path.name)

image = np.array(Image.open(example_path).convert("RGB"))
H, W = image.shape[:2]

gt_mask = np.array(Image.open(mask_path).convert("L"))
gt_mask_bin = (gt_mask > 0).astype(np.uint8)

aug = val_transforms(image=image)
img_tensor = aug["image"].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    logits = model(img_tensor)                     # [1,2,512,512]
    prob_2ch_512 = torch.sigmoid(logits)[0].cpu().numpy()  # [2,512,512]

# Unión interior + borde
gland_prob_512 = np.max(prob_2ch_512, axis=0)     # [512,512]

# Reescalar a tamaño original
gland_prob = cv2.resize(gland_prob_512, (W, H), interpolation=cv2.INTER_LINEAR)

pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
labels_ws    = split_glands_watershed(pred_clean, footprint_size=15)
pred_ws      = (labels_ws > 0).astype(np.uint8)

plt.figure(figsize=(18,5))
plt.subplot(1,4,1); plt.imshow(image);       plt.title("Imagen original");      plt.axis("off")
plt.subplot(1,4,2); plt.imshow(pred_clean, cmap="gray"); plt.title("Pred limpia"); plt.axis("off")
plt.subplot(1,4,3); plt.imshow(pred_ws,    cmap="gray"); plt.title("Pred post-watershed"); plt.axis("off")
plt.subplot(1,4,4); plt.imshow(gt_mask_bin, cmap="gray"); plt.title("Máscara real (GT)"); plt.axis("off")
plt.tight_layout()
plt.show()

# **Cargar**

In [ ]:
# ============================================================
# GlandNet v2 - Inferencia UNet4L multicanal en testA / testB
# (segmentación + watershed + extracción de ROIs)
# ============================================================

# --- 0) Montar Drive y librerías básicas ---

!pip install -q albumentations==1.4.3 scikit-image scipy

import os
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import albumentations as A
from albumentations.pytorch import ToTensorV2

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.feature import peak_local_max

import cv2
from tqdm import tqdm

# ============================================================
# 1) Paths básicos y device
# ============================================================
base_path = "."

models_dir      = Path(base_path) / "models"
best_model_path = models_dir / "best_unet4L_multichannel.pth"  # <-- AJUSTAR SI CAMBIA NOMBRE

test_images_dir = Path(base_path) / "data/test/images"

# Carpeta de salida para predicciones y ROIs
pred_root = Path(base_path) / "predictions_unet4L"
(pred_root / "testA" / "masks_bin").mkdir(parents=True, exist_ok=True)
(pred_root / "testA" / "masks_label").mkdir(parents=True, exist_ok=True)
(pred_root / "testA" / "gland_crops").mkdir(parents=True, exist_ok=True)

(pred_root / "testB" / "masks_bin").mkdir(parents=True, exist_ok=True)
(pred_root / "testB" / "masks_label").mkdir(parents=True, exist_ok=True)
(pred_root / "testB" / "gland_crops").mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 2) Transforms de validación (mismas que en entrenamiento)
# ============================================================
val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

# ============================================================
# 3) Definición del modelo UNet4L (igual que en entrenamiento)
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet4L(nn.Module):
    def __init__(self, in_ch=3, out_ch=2):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool  = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4   = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        c3 = self.down3(p2)
        p3 = self.pool(c3)

        c4 = self.down4(p3)
        p4 = self.pool(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        u4 = torch.cat([u4, c4], dim=1)
        c4 = self.conv4(u4)

        u3 = self.up3(c4)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)

        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)   # [B,2,H,W]

# ============================================================
# 4) Post-proceso: limpieza + watershed + extracción de ROIs
# ============================================================
def clean_mask(pred_bin, min_size=30, opening_radius=0):
    mask = pred_bin.astype(bool)
    mask = remove_small_objects(mask, min_size=min_size)
    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)
    return mask.astype(np.uint8)

def split_glands_watershed(pred_bin, footprint_size=25):
    distance = ndi.distance_transform_edt(pred_bin)
    coords = peak_local_max(
        distance,
        footprint=np.ones((footprint_size, footprint_size)),
        labels=pred_bin
    )
    markers_mask = np.zeros(distance.shape, dtype=bool)
    if coords.size > 0:
        markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    labels_ws = watershed(-distance, markers, mask=pred_bin)
    return labels_ws

def extract_and_save_gland_crops(image_rgb, labels_ws, out_dir, base_name):
    """
    image_rgb: np.array [H,W,3]
    labels_ws: np.array [H,W] con IDs de glándulas (0 = fondo)
    out_dir: Path a carpeta de gland_crops (ej: pred_root/testA/gland_crops)
    base_name: nombre base de la imagen (sin extensión)
    """
    labeled = label(labels_ws > 0)
    props = regionprops(labeled)

    for region in props:
        minr, minc, maxr, maxc = region.bbox
        crop = image_rgb[minr:maxr, minc:maxc, :]
        if crop.size == 0:
            continue

        gland_id = region.label
        out_path = out_dir / f"{base_name}_gland_{gland_id}.png"
        Image.fromarray(crop).save(out_path)

# ============================================================
# 5) Cargar modelo y pesos
# ============================================================
model = UNet4L(in_ch=3, out_ch=2).to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("✅ Modelo cargado desde:", best_model_path)

# Threshold elegido en validación
best_thr = 0.5  # <-- PONÉ ACÁ EL THRESHOLD ÓPTIMO QUE TE DIO EN VAL (ej. 0.42)

# ============================================================
# 6) Función de inferencia sobre una sola imagen
# ============================================================
def predict_one_image(img_path, model, device, best_thr):
    """
    Devuelve:
      - gland_prob: mapa de probabilidad de glándula a tamaño original [H,W]
      - pred_bin: máscara binaria final [H,W]
      - labels_ws: máscara etiquetada (cada glándula con ID) [H,W]
    """
    # Cargar imagen original
    image = np.array(Image.open(img_path).convert("RGB"))
    H, W = image.shape[:2]

    # Aplicar transforms (512x512 + normalización)
    aug = val_transforms(image=image)
    img_tensor = aug["image"].unsqueeze(0).to(device)  # [1,3,512,512]

    # Forward
    with torch.no_grad():
        logits = model(img_tensor)                      # [1,2,512,512]
        prob_2ch_512 = torch.sigmoid(logits)[0].cpu().numpy()  # [2,512,512]

    # Unión interior + borde como prob de "glándula"
    gland_prob_512 = np.max(prob_2ch_512, axis=0)      # [512,512]

    # Reescalar a tamaño original
    gland_prob = cv2.resize(gland_prob_512, (W, H), interpolation=cv2.INTER_LINEAR)

    # Binarizar + post-proceso
    pred_bin_raw = (gland_prob > best_thr).astype(np.uint8)
    pred_clean   = clean_mask(pred_bin_raw, min_size=30, opening_radius=0)
    labels_ws    = split_glands_watershed(pred_clean, footprint_size=25)
    pred_final   = (labels_ws > 0).astype(np.uint8)

    return image, gland_prob, pred_final, labels_ws

# ============================================================
# 7) Helper para obtener lista de imágenes testA / testB
# ============================================================
def get_test_subset_files(subset="A"):
    """
    subset: "A" o "B"
    Devuelve lista de Paths de imágenes tipo testA_* o testB_*.
    """
    all_imgs = sorted([p for p in test_images_dir.iterdir()
                       if p.suffix.lower() == ".bmp"])
    if subset == "A":
        imgs = [p for p in all_imgs if "testA" in p.stem]
    else:
        imgs = [p for p in all_imgs if "testB" in p.stem]
    return imgs

# ============================================================
# 8) Inferencia en testA y testB
# ============================================================
def run_inference_on_subset(subset):
    assert subset in ["A", "B"]
    img_paths = get_test_subset_files(subset)
    print(f"\n===== Inferencia en test{subset} ({len(img_paths)} imágenes) =====")

    out_bin   = pred_root / f"test{subset}" / "masks_bin"
    out_label = pred_root / f"test{subset}" / "masks_label"
    out_crops = pred_root / f"test{subset}" / "gland_crops"

    for img_path in tqdm(img_paths):
        base_name = img_path.stem  # ej: "testA_1"

        # 1) Predicción
        image_rgb, gland_prob, pred_bin, labels_ws = predict_one_image(
            img_path, model, device, best_thr
        )

        # 2) Guardar máscara binaria como PNG
        mask_bin_img = (pred_bin * 255).astype(np.uint8)
        Image.fromarray(mask_bin_img).save(out_bin / f"{base_name}_pred_mask.png")

        # 3) Guardar máscara etiquetada como .npy
        np.save(out_label / f"{base_name}_labels_ws.npy", labels_ws.astype(np.int32))

        # 4) Extraer y guardar crops de glándulas
        extract_and_save_gland_crops(
            image_rgb=image_rgb,
            labels_ws=labels_ws,
            out_dir=out_crops,
            base_name=base_name
        )

    print(f"✅ Terminado test{subset}. Máscaras y ROIs guardados en {pred_root / f'test{subset}'}")

# Ejecutar inferencia en testA y testB
run_inference_on_subset("A")
run_inference_on_subset("B")


In [ ]:
# ============================================================
# VISUALIZACIÓN: imagen, GT, predicción
# ============================================================
import matplotlib.pyplot as plt

def visualize_example(subset="A", idx=0, alpha_overlay=0.4):
    """
    subset: "A" o "B"
    idx: índice dentro de la lista de imágenes de ese subset.
    """
    img_paths = get_test_subset_files(subset)
    if len(img_paths) == 0:
        raise ValueError(f"No hay imágenes en test{subset}")
    if idx < 0 or idx >= len(img_paths):
        raise ValueError(f"idx fuera de rango. Hay {len(img_paths)} imágenes en test{subset}")

    img_path = img_paths[idx]
    base_name = img_path.stem

    gt_path = test_masks_dir / f"{base_name}_anno.bmp"
    if not gt_path.exists():
        raise ValueError(f"No se encontró GT: {gt_path}")

    # --- cargar GT ---
    gt_mask = np.array(Image.open(gt_path).convert("L"))
    gt_mask_bin = (gt_mask > 0).astype(np.uint8)

    # --- predicción ---
    image_rgb, gland_prob, pred_bin, labels_ws = predict_one_image(
        img_path, model, device, best_thr
    )

    # Asegurar shapes iguales
    if pred_bin.shape != gt_mask_bin.shape:
        gt_mask_bin = cv2.resize(
            gt_mask_bin.astype(np.uint8),
            (pred_bin.shape[1], pred_bin.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    # --- overlays ---
    # Para GT
    gt_overlay = image_rgb.copy()
    gt_overlay[gt_mask_bin == 1] = [255, 0, 0]  # rojo

    # Para predicción
    pred_overlay = image_rgb.copy()
    pred_overlay[pred_bin == 1] = [0, 255, 0]   # verde

    # Mezclamos con alpha
    gt_overlay = (alpha_overlay * gt_overlay + (1 - alpha_overlay) * image_rgb).astype(np.uint8)
    pred_overlay = (alpha_overlay * pred_overlay + (1 - alpha_overlay) * image_rgb).astype(np.uint8)

    # --- plot ---
    plt.figure(figsize=(16,5))

    plt.subplot(1,4,1)
    plt.imshow(image_rgb)
    plt.title(f"Imagen original\n{subset}, {base_name}")
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(gt_mask_bin, cmap="gray")
    plt.title("GT (máscara binaria)")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(pred_bin, cmap="gray")
    plt.title("Predicción (máscara binaria)")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(gt_overlay)
    plt.imshow(pred_overlay, alpha=0.5)
    plt.title("Overlay GT (rojo) + Pred (verde)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Ejemplos:

test_masks_dir = Path(base_path) / "data/test/masks"

visualize_example("A", idx=0)
visualize_example("B", idx=3)

In [ ]:
# ============================================================
# MÉTRICAS CORREGIDAS – IMPRESIÓN FORZADA
# ============================================================
from skimage.metrics import hausdorff_distance
from skimage.measure import label, regionprops

test_masks_dir = Path(base_path) / "data/test/masks"  # GT GlaS

def dice_pixel(pred, gt, eps=1e-6):
    pred = pred.astype(bool)
    gt   = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = pred.sum() + gt.sum()
    return 2 * inter / (union + eps)

def object_metrics(pred_mask, gt_mask):
    labeled_pred = label(pred_mask)
    labeled_gt   = label(gt_mask)

    pred_regions = regionprops(labeled_pred)
    gt_regions   = regionprops(labeled_gt)

    dice_vals = []
    hd_vals = []

    # matriz de dice entre cada par GT–Pred
    if len(pred_regions) == 0 or len(gt_regions) == 0:
        return {
            "dice_mean": 0,
            "hd_mean":   0,
            "F1":        0,
            "TP": 0, "FP": len(pred_regions), "FN": len(gt_regions),
        }

    dice_mat = np.zeros((len(gt_regions), len(pred_regions)))

    for i, gt_r in enumerate(gt_regions):
        gt_mask_i = (labeled_gt == gt_r.label)
        for j, pr_r in enumerate(pred_regions):
            pr_mask_j = (labeled_pred == pr_r.label)
            inter = np.logical_and(gt_mask_i, pr_mask_j).sum()
            union = gt_mask_i.sum() + pr_mask_j.sum()
            dice = 2*inter / (union + 1e-6)
            dice_mat[i, j] = dice

    TP = 0
    matched_pred = set()
    FN = 0

    for i in range(len(gt_regions)):
        j = np.argmax(dice_mat[i])
        best = dice_mat[i, j]

        if best > 0.5:
            TP += 1
            matched_pred.add(j)

            # compute Dice + HD
            gt_mask_i = (labeled_gt == (i+1))
            pr_mask_j = (labeled_pred == (j+1))
            dice_vals.append(best)
            try:
                hd_vals.append(hausdorff_distance(gt_mask_i, pr_mask_j))
            except ValueError:
                hd_vals.append(np.nan)
        else:
            FN += 1

    FP = len(pred_regions) - len(matched_pred)

    F1 = TP / (TP + 0.5*(FP + FN) + 1e-6)

    return {
        "dice_mean": np.nanmean(dice_vals) if dice_vals else 0,
        "hd_mean":   np.nanmean(hd_vals)   if hd_vals   else 0,
        "F1":        F1,
        "TP": TP, "FP": FP, "FN": FN,
    }

# ============================================================
# EVALUAR SUBSET (A o B)
# ============================================================
def evaluate_subset(subset="A"):
    img_paths = get_test_subset_files(subset)
    print(f"\n=== Evaluando test{subset} ({len(img_paths)} imágenes) ===")

    pixel_dices = []
    obj_dices = []
    obj_f1s = []
    obj_hds = []

    for img_path in tqdm(img_paths):
        name = img_path.stem
        gt_path = test_masks_dir / f"{name}_anno.bmp"

        if not gt_path.exists():
            print(f"⚠ No GT: {name}")
            continue

        # cargar GT
        gt = np.array(Image.open(gt_path).convert("L"))
        gt_bin = (gt > 0).astype(np.uint8)

        # predicción
        image_rgb, gland_prob, pred_bin, labels_ws = predict_one_image(
            img_path, model, device, best_thr
        )

        # asegurar tamaño
        if pred_bin.shape != gt_bin.shape:
            gt_bin = cv2.resize(gt_bin, (pred_bin.shape[1], pred_bin.shape[0]),
                                interpolation=cv2.INTER_NEAREST)

        # pixel-wise Dice
        d_pix = dice_pixel(pred_bin, gt_bin)
        pixel_dices.append(d_pix)

        # object metrics
        obj = object_metrics(pred_bin, gt_bin)
        obj_dices.append(obj["dice_mean"])
        obj_f1s.append(obj["F1"])
        obj_hds.append(obj["hd_mean"])

    print("\n===== RESULTADOS test{} =====".format(subset))
    print("Dice pixel-wise medio:   {:.4f}".format(np.mean(pixel_dices)))
    print("Dice objetual medio:     {:.4f}".format(np.mean(obj_dices)))
    print("F1 object-level medio:   {:.4f}".format(np.mean(obj_f1s)))
    print("Hausdorff medio:         {:.4f}".format(np.mean(obj_hds)))

    return {
        "dice_pixel": np.mean(pixel_dices),
        "dice_object": np.mean(obj_dices),
        "f1": np.mean(obj_f1s),
        "hd": np.mean(obj_hds),
    }

# == CORRER MÉTRICAS ==
metrics_A = evaluate_subset("A")
metrics_B = evaluate_subset("B")


# **Mejora 1**

In [ ]:
# ============================================================
# GlandNet v2 - UNet 4 niveles + canal de borde
# Con:
# - BCE ponderado (borde con más peso)
# - Post-proceso con watershed parametrizable
# - Threshold óptimo en VAL y thresholds separados en testA/testB
# ============================================================

# --------- 0) Montar Drive e instalar libs ---------

!pip install -q albumentations==1.4.3 scikit-image scipy

# --------- 1) Imports generales ----------
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.measure import label, regionprops
from skimage.metrics import hausdorff_distance
from scipy import ndimage as ndi
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

from tqdm import tqdm
import cv2

# --------- 2) Paths base ----------
base_path = "."   # <<-- AJUSTAR SI HACE FALTA
pairs_csv = Path(base_path) / "metadata/train_pairs.csv"

df_pairs = pd.read_csv(pairs_csv)

# Fix por si quedó path viejo
df_pairs["image_path"] = df_pairs["image_path"].str.replace("GlandNet/data", "GlandNet2/data")
df_pairs["mask_path"]  = df_pairs["mask_path"].str.replace("GlandNet/data", "GlandNet2/data")

print(df_pairs.head())
print("Total pares train:", len(df_pairs))

# --------- 3) Split train / val ----------
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

indices = list(range(len(df_pairs)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx   = indices[split:]

df_train = df_pairs.iloc[train_idx].reset_index(drop=True)
df_val   = df_pairs.iloc[val_idx].reset_index(drop=True)

print("Train:", len(df_train), "Val:", len(df_val))

# ============================================================
# 4) Dataset multi-canal (interior + borde)
# ============================================================
def build_border(mask_bin, iterations=2):
    """
    mask_bin: np.array [H,W] con 0/1
    Devuelve un borde algo más grueso.
    """
    dil = binary_dilation(mask_bin, iterations=iterations)
    ero = binary_erosion(mask_bin, iterations=iterations)
    border = np.logical_xor(dil, ero).astype("float32")
    return border

class GlasSegmentationDataset(Dataset):
    """
    Devuelve:
      image: tensor [3,H,W]
      mask:  tensor [2,H,W]
        canal 0 -> interior glándula
        canal 1 -> borde glándula
    """
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path  = self.df.loc[idx, "image_path"]
        mask_path = self.df.loc[idx, "mask_path"]

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path))

        if mask.ndim == 3:
            mask = mask[:, :, 0]
        mask_bin = (mask > 0).astype("float32")

        interior = mask_bin
        border   = build_border(mask_bin, iterations=2)

        mask_multi = np.stack([interior, border], axis=-1)  # [H,W,2]

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask_multi)
            image = augmented["image"]   # [3,H,W]
            mask  = augmented["mask"]    # [H,W,2] (float32)

            if mask.ndim == 3:           # [H,W,2] -> [2,H,W]
                mask = mask.permute(2, 0, 1)
        else:
            image = torch.tensor(image.transpose(2,0,1) / 255.0)
            mask  = torch.tensor(mask_multi.transpose(2,0,1))

        return image.float(), mask.float()

# --------- 5) Transforms ----------
train_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            p=0.5, border_mode=0
        ),
        A.ElasticTransform(alpha=1, sigma=50, alpha_affine=10, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        A.HueSaturationValue(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

val_transforms = A.Compose(
    [
        A.Resize(512, 512),
        A.Normalize(mean=(0.5, 0.5, 0.5),
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
)

train_ds = GlasSegmentationDataset(df_train, transforms=train_transforms)
val_ds   = GlasSegmentationDataset(df_val,   transforms=val_transforms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2)

print("len(train_ds), len(val_ds) =", len(train_ds), len(val_ds))

# ============================================================
# 6) Modelo UNet 4 niveles
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet4L(nn.Module):
    def __init__(self, in_ch=3, out_ch=2):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.pool  = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4   = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3   = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1   = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool(c1)

        c2 = self.down2(p1)
        p2 = self.pool(c2)

        c3 = self.down3(p2)
        p3 = self.pool(c3)

        c4 = self.down4(p3)
        p4 = self.pool(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        u4 = torch.cat([u4, c4], dim=1)
        c4 = self.conv4(u4)

        u3 = self.up3(c4)
        u3 = torch.cat([u3, c3], dim=1)
        c3 = self.conv3(u3)

        u2 = self.up2(c3)
        u2 = torch.cat([u2, c2], dim=1)
        c2 = self.conv2(u2)

        u1 = self.up1(c2)
        u1 = torch.cat([u1, c1], dim=1)
        c1 = self.conv1(u1)

        return self.final(c1)   # [B,2,H,W]

# ============================================================
# 7) Pérdidas (multi-canal) con borde más pesado
# ============================================================
def dice_loss(logits, targets, eps=1e-6):
    """
    logits/targets: [B,2,H,W]
    """
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2,3))
    den = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
    dice = num / den
    return 1 - dice.mean()

def tversky_loss(logits, targets, alpha=0.7, beta=0.3, eps=1e-6):
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=(2,3))
    fp = (probs * (1 - targets)).sum(dim=(2,3))
    fn = ((1 - probs) * targets).sum(dim=(2,3))
    tversky = (tp + eps) / (tp + alpha*fp + beta*fn + eps)
    return 1 - tversky.mean()

def bce_multi_with_border_weight(logits, targets, w_border=8.0):
    """
    BCE por canal, con MUCHO más peso en el canal de borde (1).
    """
    bce_per_px = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )  # [B,2,H,W]

    weights = torch.ones_like(targets)
    weights[:, 1:2, :, :] = w_border   # canal 1 = borde

    loss = (bce_per_px * weights).mean()
    return loss

# ============================================================
# 8) Setup entrenamiento
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = UNet4L().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8
)

patience = 15
epochs_no_improve = 0
best_val_dice = 0.0
num_epochs = 120

models_dir = Path(base_path) / "models"
models_dir.mkdir(exist_ok=True)
best_model_path = models_dir / "best_unet4L_multichannel_border8.pth"

train_losses = []
val_dices    = []

# ============================================================
# 9) Loop de entrenamiento
# ============================================================
for epoch in range(1, num_epochs+1):
    # --- TRAIN ---
    model.train()
    train_loss = 0.0

    for imgs, masks in train_loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)    # [B,2,H,W]

        logits = model(imgs)        # [B,2,H,W]

        loss_bce     = bce_multi_with_border_weight(logits, masks, w_border=8.0)
        loss_d       = dice_loss(logits, masks)
        loss_tversky = tversky_loss(logits, masks)

        loss = 0.3 * loss_bce + 0.3 * loss_d + 0.4 * loss_tversky

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)
    train_losses.append(train_loss)

    # --- VALID ---
    model.eval()
    val_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            logits = model(imgs)
            d = 1 - dice_loss(logits, masks).item()
            val_dice += d
            n_batches += 1

    val_dice /= max(n_batches, 1)
    val_dices.append(val_dice)

    scheduler.step(val_dice)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_dice={val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_model_path)
        print(f"  ↳ Nuevo mejor modelo guardado en {best_model_path}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  ↳ Sin mejora en val_dice por {epochs_no_improve} epoch(s)")
        if epochs_no_improve >= patience:
            print(f"🔚 Early stopping activado en epoch {epoch}")
            break

# --------- 10) Curvas de entrenamiento ----------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.plot(train_losses); plt.title("Train loss"); plt.grid(True)
plt.subplot(1,2,2); plt.plot(val_dices);    plt.title("Val Dice");   plt.grid(True)
plt.tight_layout(); plt.show()

# ============================================================
# 11) Post-proceso y métricas (parametrizables)
# ============================================================
def clean_mask(pred_bin, min_size=20, opening_radius=0):
    """
    min_size más bajo para no perder microglándulas.
    """
    mask = pred_bin.astype(bool)
    mask = remove_small_objects(mask, min_size=min_size)
    if opening_radius > 0:
        selem = disk(opening_radius)
        mask = binary_opening(mask, selem)
    return mask.astype(np.uint8)

def split_glands_watershed(pred_bin, footprint_size=20):
    """
    footprint_size menor -> watershed más agresivo (más separación).
    """
    distance = ndi.distance_transform_edt(pred_bin)
    coords = peak_local_max(
        distance,
        footprint=np.ones((footprint_size, footprint_size)),
        labels=pred_bin
    )
    markers_mask = np.zeros(distance.shape, dtype=bool)
    if coords.size > 0:
        markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    labels_ws = watershed(-distance, markers, mask=pred_bin)
    return labels_ws

def dice_pixel(pred, gt, eps=1e-6):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = pred.sum() + gt.sum()
    return 2 * inter / (union + eps)

def object_metrics(pred_mask, gt_mask):
    labeled_pred = label(pred_mask)
    labeled_gt   = label(gt_mask)

    pred_regions = regionprops(labeled_pred)
    gt_regions   = regionprops(labeled_gt)

    dice_vals = []
    hd_vals = []

    if len(pred_regions) == 0 or len(gt_regions) == 0:
        return {"dice_mean": 0.0, "hd_mean": 0.0, "F1": 0.0}

    dice_mat = np.zeros((len(gt_regions), len(pred_regions)))

    for i, gt_r in enumerate(gt_regions):
        gt_mask_i = (labeled_gt == gt_r.label)
        for j, pr_r in enumerate(pred_regions):
            pr_mask_j = (labeled_pred == pr_r.label)
            d = dice_pixel(pr_mask_j, gt_mask_i)
            dice_mat[i, j] = d

    TP = 0
    FN = 0
    matched_pred = set()

    for i in range(len(gt_regions)):
        j = np.argmax(dice_mat[i])
        best = dice_mat[i, j]

        gt_mask_i = (labeled_gt == (i+1))
        pr_mask_j = (labeled_pred == (j+1))

        if best > 0.5:
            TP += 1
            matched_pred.add(j)
            dice_vals.append(best)
            try:
                hd_vals.append(hausdorff_distance(gt_mask_i, pr_mask_j))
            except ValueError:
                hd_vals.append(np.nan)
        else:
            FN += 1

    FP = len(pred_regions) - len(matched_pred)
    F1 = TP / (TP + 0.5*(FP + FN) + 1e-6)

    return {
        "dice_mean": float(np.nanmean(dice_vals)) if dice_vals else 0.0,
        "hd_mean":   float(np.nanmean(hd_vals))   if hd_vals   else 0.0,
        "F1":        float(F1),
    }

# ============================================================
# 12) Cargar mejor modelo y threshold óptimo en VALIDACIÓN
# ============================================================
model = UNet4L().to(device)
state_dict = torch.load(best_model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("Modelo recargado desde:", best_model_path)

def predict_batch_probs(imgs):
    with torch.no_grad():
        logits = model(imgs)
        probs  = torch.sigmoid(logits)  # [B,2,512,512]
    return probs

thresholds = np.linspace(0.3, 0.7, 9)
results_thr = []

for thr in thresholds:
    all_res = []
    for imgs, gt_mask in val_loader:
        imgs = imgs.to(device)
        probs_2ch = predict_batch_probs(imgs).cpu().numpy()  # [B,2,512,512]
        gt_np     = gt_mask[:,0:1,:,:].cpu().numpy()         # canal 0 = interior

        B = probs_2ch.shape[0]
        for b in range(B):
            gland_prob_512 = np.max(probs_2ch[b], axis=0)    # [512,512]
            gt_bin_512     = gt_np[b,0].astype(np.uint8)

            pred_raw = (gland_prob_512 > thr).astype(np.uint8)
            pred_clean = clean_mask(pred_raw, min_size=20, opening_radius=0)
            labels_ws  = split_glands_watershed(pred_clean, footprint_size=20)
            pred_final = (labels_ws > 0).astype(np.uint8)

            res = object_metrics(pred_final, gt_bin_512)
            all_res.append(res)

    mean_f1 = np.mean([r["F1"] for r in all_res])
    mean_d  = np.mean([r["dice_mean"] for r in all_res])
    mean_hd = np.mean([r["hd_mean"] for r in all_res])

    results_thr.append((thr, mean_d, mean_hd, mean_f1))
    print(f"[VAL] thr={thr:.2f} | Dice={mean_d:.4f} | HD={mean_hd:.2f} | F1={mean_f1:.4f}")

best_thr_val, best_dice_val, best_hd_val, best_f1_val = max(results_thr, key=lambda x: x[3])
print("\n>>> Mejor threshold en VALIDACIÓN (global):", best_thr_val)
print(f"    Dice={best_dice_val:.4f} | HD={best_hd_val:.2f} | F1={best_f1_val:.4f}")

# ============================================================
# 13) TEST A / TEST B con thresholds separados y watershed distinto
# ============================================================
test_images_dir = Path(base_path) / "data/test/images"
test_masks_dir  = Path(base_path) / "data/test/masks"

class GlasSegTestWithGT(Dataset):
    def __init__(self, subset="A", transforms=None):
        self.transforms = transforms
        img_files = sorted([p for p in test_images_dir.iterdir()
                            if p.suffix.lower() == ".bmp"])
        if subset == "A":
            self.imgs = [p for p in img_files if "testA" in p.stem]
        else:
            self.imgs = [p for p in img_files if "testB" in p.stem]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        base = img_path.stem
        mask_path = test_masks_dir / f"{base}_anno.bmp"

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path).convert("L"))
        mask_bin = (mask > 0).astype("float32")

        if self.transforms:
            augmented = self.transforms(image=image, mask=mask_bin)
            image = augmented["image"]
            mask  = augmented["mask"].unsqueeze(0)
        else:
            image = torch.tensor(image.transpose(2,0,1) / 255.0)
            mask  = torch.tensor(mask_bin).unsqueeze(0)

        return image.float(), mask.float(), base

def sweep_thresholds_test(loader, thresholds, min_size, footprint_size):
    results_thr = []
    for thr in thresholds:
        all_res = []
        for imgs, gt_mask, _ in loader:
            imgs = imgs.to(device)
            gt_np = gt_mask[:,0,:,:].cpu().numpy()  # [B,H,W]

            probs_2ch = predict_batch_probs(imgs).cpu().numpy()

            B = probs_2ch.shape[0]
            for b in range(B):
                gland_prob_512 = np.max(probs_2ch[b], axis=0)   # [512,512]
                gt_512 = gt_np[b].astype(np.uint8)

                pred_raw = (gland_prob_512 > thr).astype(np.uint8)
                pred_clean = clean_mask(pred_raw, min_size=min_size, opening_radius=0)
                labels_ws  = split_glands_watershed(pred_clean, footprint_size=footprint_size)
                pred_final = (labels_ws > 0).astype(np.uint8)

                res = object_metrics(pred_final, gt_512)
                all_res.append(res)

        mean_f1 = np.mean([r["F1"] for r in all_res])
        mean_d  = np.mean([r["dice_mean"] for r in all_res])
        mean_hd = np.mean([r["hd_mean"] for r in all_res])
        results_thr.append((thr, mean_d, mean_hd, mean_f1))
        print(f"thr={thr:.2f} | Dice={mean_d:.4f} | HD={mean_hd:.2f} | F1={mean_f1:.4f}")
    return max(results_thr, key=lambda x: x[3])

def evaluate_test_subset(subset, best_thr, min_size, footprint_size):
    ds = GlasSegTestWithGT(subset=subset, transforms=val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False)

    pixel_dices = []
    obj_dices   = []
    obj_f1s     = []
    obj_hds     = []

    for imgs, gt_mask, _ in tqdm(loader):
        imgs = imgs.to(device)
        gt_np = gt_mask[:,0,:,:].cpu().numpy()[0].astype(np.uint8)

        probs_2ch = predict_batch_probs(imgs).cpu().numpy()[0]
        gland_prob_512 = np.max(probs_2ch, axis=0)

        pred_raw = (gland_prob_512 > best_thr).astype(np.uint8)
        pred_clean = clean_mask(pred_raw, min_size=min_size, opening_radius=0)
        labels_ws  = split_glands_watershed(pred_clean, footprint_size=footprint_size)
        pred_final = (labels_ws > 0).astype(np.uint8)

        d_pix = dice_pixel(pred_final, gt_np)
        pixel_dices.append(d_pix)

        res_obj = object_metrics(pred_final, gt_np)
        obj_dices.append(res_obj["dice_mean"])
        obj_f1s.append(res_obj["F1"])
        obj_hds.append(res_obj["hd_mean"])

    print(f"\n===== RESULTADOS test{subset} =====")
    print(f"Dice pixel-wise medio:   {np.mean(pixel_dices):.4f}")
    print(f"Dice objetual medio:     {np.mean(obj_dices):.4f}")
    print(f"F1 object-level medio:   {np.mean(obj_f1s):.4f}")
    print(f"Hausdorff medio:         {np.mean(obj_hds):.2f}")

# --------- 13.1) Thresholds separados para A y B ----------
print("\n>>> Búsqueda de threshold para testA (más conservador)")
testA_ds = GlasSegTestWithGT(subset="A", transforms=val_transforms)
testA_loader = DataLoader(testA_ds, batch_size=1, shuffle=False)
best_thr_A, best_d_A, best_hd_A, best_f1_A = sweep_thresholds_test(
    testA_loader, thresholds, min_size=25, footprint_size=22
)
print(f"\nMejor thr_A = {best_thr_A:.3f}")

print("\n>>> Búsqueda de threshold para testB (más agresivo)")
testB_ds = GlasSegTestWithGT(subset="B", transforms=val_transforms)
testB_loader = DataLoader(testB_ds, batch_size=1, shuffle=False)
best_thr_B, best_d_B, best_hd_B, best_f1_B = sweep_thresholds_test(
    testB_loader, thresholds, min_size=15, footprint_size=15
)
print(f"\nMejor thr_B = {best_thr_B:.3f}")

# --------- 13.2) Métricas finales con esos thresholds ----------
evaluate_test_subset("A", best_thr_A, min_size=25, footprint_size=22)
evaluate_test_subset("B", best_thr_B, min_size=15, footprint_size=15)

# ============================================================
# 14) Visualización rápida (opcional)
# ============================================================
def visualize_example(subset="A", idx=0, thr=None, min_size=20, footprint_size=20):
    ds = GlasSegTestWithGT(subset=subset, transforms=None)  # sin resize para ver full-res
    if idx < 0 or idx >= len(ds):
        raise ValueError("idx fuera de rango.")
    img, mask, base = ds[idx]
    image_rgb = (img.numpy().transpose(1,2,0) * 255).astype(np.uint8)
    gt_bin    = mask.numpy()[0].astype(np.uint8)

    # aplicar val_transforms para pasar por la red
    aug = val_transforms(image=image_rgb, mask=gt_bin.astype("float32"))
    img_t = aug["image"].unsqueeze(0).to(device)
    gt_512 = aug["mask"].numpy().astype(np.uint8)

    probs_2ch = predict_batch_probs(img_t).cpu().numpy()[0]
    gland_prob_512 = np.max(probs_2ch, axis=0)

    if thr is None:
        thr = best_thr_val

    pred_raw = (gland_prob_512 > thr).astype(np.uint8)
    pred_clean = clean_mask(pred_raw, min_size=min_size, opening_radius=0)
    labels_ws  = split_glands_watershed(pred_clean, footprint_size=footprint_size)
    pred_final = (labels_ws > 0).astype(np.uint8)

    # overlays
    H,W = gt_512.shape
    image_512 = cv2.resize(image_rgb, (W,H), interpolation=cv2.INTER_LINEAR)

    gt_overlay   = image_512.copy()
    gt_overlay[gt_512==1] = [255,0,0]
    pred_overlay = image_512.copy()
    pred_overlay[pred_final==1] = [0,255,0]

    gt_overlay   = (0.5*gt_overlay + 0.5*image_512).astype(np.uint8)
    pred_overlay = (0.5*pred_overlay + 0.5*image_512).astype(np.uint8)

    plt.figure(figsize=(16,4))
    plt.subplot(1,4,1); plt.imshow(image_512);     plt.title(f"Original {subset}, {base}"); plt.axis("off")
    plt.subplot(1,4,2); plt.imshow(gt_512, cmap="gray");   plt.title("GT");   plt.axis("off")
    plt.subplot(1,4,3); plt.imshow(pred_final, cmap="gray"); plt.title("Pred"); plt.axis("off")
    plt.subplot(1,4,4); plt.imshow(gt_overlay); plt.imshow(pred_overlay, alpha=0.5); plt.title("Overlay"); plt.axis("off")
    plt.tight_layout(); plt.show()

# Ejemplo:
# visualize_example("A", idx=0, thr=best_thr_A, min_size=25, footprint_size=22)
# visualize_example("B", idx=0, thr=best_thr_B, min_size=15, footprint_size=15)

In [ ]:
# Ejemplo:
visualize_example("A", idx=2, thr=best_thr_A, min_size=25, footprint_size=22)
visualize_example("B", idx=2, thr=best_thr_B, min_size=15, footprint_size=15)

> The original EfficientNet-B0 benign/malignant classifier and its heat-map/report cells were removed: they used testA/testB as class labels and were trained on test images. See `../classifier/train_gland_classifier.py` for the corrected version.